In [5]:
import numpy as np

# как из файла generdat.m
def generdat(N, V, k, alpha, nmin):
    # generdat.py Generation of NxV data matrix consisting of k Gaussian clusters
    # containing at least nmin elements each. with squeezing parameter alpha

    # ----------------generating cluster cardinalities randomly
    nn1 = np.ones(k, dtype=int)
    # nmin=4
    while np.min(nn1) < nmin:
        N0 = N
        nn = np.ones(k, dtype=int)
        for k0 in range(1, k):
            if N0 > 1:

                n0 = np.random.randint(1, N0)
                N0 = N0 - n0
                nn[k0-1] = n0
            else:
                nn[k0-1] = 2
        nn[k-1] = N - np.sum(nn) + 1
        nn1 = nn
    Nk = nn1
    # -------------------------generating cluster centers
    cen = (alpha - 1) + 2 * (1 - alpha) * np.random.rand(k, V)
    # --------generating clusters---------------
    X = np.empty((0, V))
    count = 1
    R = [None] * k
    rf = np.zeros(N, dtype=int)
    for k0 in range(1, k+1):
        nk = Nk[k0-1]  # cluster size
        R[k0-1] = np.arange(count, count + nk)
        rf[R[k0-1] - 1] = k0
        count = count + nk
        sig = 0.05 + 0.05 * np.random.rand(V)  # randomness in sigmas
        Xk = np.random.randn(nk, V)
        Xk = Xk * np.tile(sig, (nk, 1))
        Xk = Xk + np.tile(cen[k0-1, :], (nk, 1))  # matrix for cluster k
        X = np.vstack((X, Xk))
    return Nk, R, rf, X, cen

ГЕНЕРАЦИЯ ВСЕХ ДАТАСЕТОВ (по статье)

In [ ]:
from pathlib import Path
from joblib import Parallel, delayed
import os
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt


def _sample_cluster_sizes_fast(N, k, nmin, rng):
    # каждому кластеру минимум nmin, остаток распределяем случайно
    if k * nmin > N:
        raise ValueError(f"Требуется k*nmin <= N, получено {k*nmin} > {N}")

    remainder = N - k * nmin
    extras = rng.multinomial(remainder, np.full(k, 1.0 / k))
    return extras + nmin


def generdat_fast(N, V, k, alpha, nmin, rng=None, use_gpu=False):
    rng = np.random.default_rng() if rng is None else rng
    Nk = _sample_cluster_sizes_fast(N, k, nmin, rng)

    counts = Nk.astype(int)
    starts = np.cumsum(np.r_[0, counts[:-1]])
    ends = np.cumsum(counts)
    R = [np.arange(s + 1, e + 1) for s, e in zip(starts, ends)]
    rf = np.repeat(np.arange(1, k + 1), counts)

    if use_gpu:
        try:
            import cupy as cp
        except ImportError as exc:
            raise ImportError(
                "GPU режим требует cupy. Установите подходящий пакет, например: pip install cupy-cuda12x"
            ) from exc

        cp.random.seed(int(rng.integers(0, 2**31 - 1)))

        cen_gpu = (alpha - 1) + 2 * (1 - alpha) * cp.random.random((k, V))
        blocks = []
        for i, nk in enumerate(counts):
            sig = 0.05 + 0.05 * cp.random.random(V)
            Xk = cp.random.standard_normal((nk, V))
            Xk = Xk * sig + cen_gpu[i]
            blocks.append(Xk)

        X = cp.asnumpy(cp.vstack(blocks))
        cen = cp.asnumpy(cen_gpu)
    else:
        cen = (alpha - 1) + 2 * (1 - alpha) * rng.random((k, V))
        blocks = []
        for i, nk in enumerate(counts):
            sig = 0.05 + 0.05 * rng.random(V)
            Xk = rng.standard_normal((nk, V))
            Xk = Xk * sig + cen[i]
            blocks.append(Xk)
        X = np.vstack(blocks)

    return Nk, R, rf, X, cen


def _generate_one_combo_fast(
    combo_dir,
    N,
    V,
    true_k,
    alpha,
    nmin,
    n_repeats,
    save_data,
    generate_plots,
    seed,
    use_gpu,
):
    combo_dir = Path(combo_dir)
    combo_dir.mkdir(parents=True, exist_ok=True)

    min_size, max_size = None, None
    for rep in range(1, n_repeats + 1):
        rng = np.random.default_rng(seed + rep)
        Nk, R, rf, X, cen = generdat_fast(
            N=N,
            V=V,
            k=true_k,
            alpha=alpha,
            nmin=nmin,
            rng=rng,
            use_gpu=use_gpu,
        )

        if X.shape != (N, V):
            raise RuntimeError("Неверная размерность X")

        min_size = int(np.min(Nk))
        max_size = int(np.max(Nk))

        if save_data:
            np.save(combo_dir / f"rep{rep:03d}_X.npy", X)
            np.save(combo_dir / f"rep{rep:03d}_rf.npy", rf)
            np.save(combo_dir / f"rep{rep:03d}_cen.npy", cen)

        # графики
        if generate_plots:
            X2d = PCA(n_components=2).fit_transform(X)
            plt.figure(figsize=(8, 6))
            for label in np.unique(rf):
                idx = rf == label
                plt.scatter(X2d[idx, 0], X2d[idx, 1], s=18, alpha=0.65, edgecolors="none")
            plt.title(f"V={V}, K*={true_k}, alpha={alpha:.2f}, rep={rep:03d}")
            plt.tight_layout()
            plt.savefig(combo_dir / f"rep{rep:03d}_fig2_style.png", dpi=100)
            plt.close()

    return {
        "combo": f"V={V:2d}_K={true_k:2d}_alpha={alpha:.2f}",
        "min_size": min_size,
        "max_size": max_size,
        "generated": n_repeats,
    }


def run_full_experiment_fast(
    base_dir="synthetic_datasets",
    N=1000,
    nmin=20,
    n_repeats=30,
    V_list=(15, 50),
    K_list=(7, 15, 21),
    alpha_list=(0.25, 0.50, 0.75),
    save_data=True,
    generate_plots=True,
    n_jobs=-1,
    seed=42,
    use_gpu=False,
    print_summary=True,
):

    root_dir = Path(base_dir) / f"experiment"
    root_dir.mkdir(parents=True, exist_ok=True)

    combos = [(V, k, a) for V in V_list for k in K_list for a in alpha_list]
    total = len(combos)

    if print_summary:
        print(f"Всего комбинаций: {total}")
        print(f"Датасетов на комбинацию: {n_repeats}")
        print(f"Всего будет сгенерировано: {total * n_repeats} наборов")
        print(f"Папка результата: {root_dir}\n")

    if n_jobs == -1:
        cpu_cnt = os.cpu_count() or 4
        n_jobs = max(1, cpu_cnt - 1)

    if use_gpu and n_jobs != 1:
        if print_summary:
            print("use_gpu=True - для стабильности n_jobs=1")
        n_jobs = 1

    tasks = []
    for idx, (V, true_k, alpha) in enumerate(combos):
        combo_str = f"V={V:2d}_K={true_k:2d}_alpha={alpha:.2f}"
        combo_dir = root_dir / combo_str
        combo_seed = seed + idx * 100000
        tasks.append(
            (
                combo_dir,
                N,
                V,
                true_k,
                alpha,
                nmin,
                n_repeats,
                save_data,
                generate_plots,
                combo_seed,
                use_gpu,
            )
        )

    results = Parallel(n_jobs=n_jobs, backend="loky", verbose=0)(
        delayed(_generate_one_combo_fast)(*args) for args in tasks
    )

    summary_path = root_dir / "summary.txt"
    with open(summary_path, "w", encoding="utf-8") as f:
        f.write(f"N={N}, nmin={nmin}, repeats={n_repeats}, n_jobs={n_jobs}, use_gpu={use_gpu}\n")
        for r in sorted(results, key=lambda x: x["combo"]):
            f.write(
                f"{r['combo']:25s} | размеры: {r['min_size']:3d}-{r['max_size']:4d} | сгенерировано: {r['generated']}\n"
            )

    return str(root_dir)


# быстрый запуск (с визуализациями для каждой комбинации)
root_dir = run_full_experiment_fast(
    base_dir="synthetic_datasets",
    N=1000,
    nmin=20,
    n_repeats=30,
    generate_plots=True,
    n_jobs=-1,
    use_gpu=False,
)
print(root_dir)

Всего комбинаций: 18
Датасетов на комбинацию: 30
Всего будет сгенерировано: 540 наборов
Папка результата: synthetic_datasets/experiment

synthetic_datasets/experiment


In [4]:
!pip install hdbscan

Кластеризация идет по объектам 

In [ ]:
from pathlib import Path
import itertools
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score


def load_generated_datasets(experiment_dir):
    # Собирает пары (X, rf) из всех подпапок эксперимента.
    # rf — это истинные метки кластеров из генератора.
    experiment_dir = Path(experiment_dir)
    if not experiment_dir.exists():
        raise FileNotFoundError(f"Папка эксперимента не найдена: {experiment_dir}")

    records = []
    for combo_dir in sorted([p for p in experiment_dir.iterdir() if p.is_dir()]):
        x_files = sorted(combo_dir.glob("rep*_X.npy"))
        for x_path in x_files:
            rep_id = x_path.stem.replace("_X", "")
            rf_path = combo_dir / f"{rep_id}_rf.npy"
            if not rf_path.exists():
                continue

            X = np.load(x_path)
            y_true = np.load(rf_path)
            records.append(
                {
                    "combo": combo_dir.name,
                    "rep": rep_id,
                    "x_path": str(x_path),
                    "rf_path": str(rf_path),
                    "X": X,
                    "y_true": y_true,
                }
            )

    if not records:
        raise ValueError(f"В {experiment_dir} не найдены пары rep*_X.npy + rep*_rf.npy")

    return records


def evaluate_predictions(y_true, y_pred):
    # Возвращает метрики и служебные показатели.
    # Шумовые точки (label = -1) учитываются как отдельный класс.
    n_noise = int(np.sum(y_pred == -1))
    n_clusters = len(set(y_pred)) - (1 if -1 in y_pred else 0)

    # Если алгоритм пометил всё как один кластер/шум, метрики всё равно считаем.
    ari = adjusted_rand_score(y_true, y_pred)
    nmi = normalized_mutual_info_score(y_true, y_pred)

    return {
        "ari": ari,
        "nmi": nmi,
        "pred_clusters": n_clusters,
        "noise_points": n_noise,
    }


def summarize_results(df, score_col="ari"):
    return (
        df.groupby("combo")
        .agg(
            **{
                f"{score_col}_mean": (score_col, "mean"),
                f"{score_col}_std": (score_col, "std"),
                f"{score_col}_min": (score_col, "min"),
                f"{score_col}_max": (score_col, "max"),
            }
        )
        .reset_index()
    )

In [ ]:
from pathlib import Path
from joblib import Parallel, delayed
import os
import itertools
import pandas as pd


from sklearn.cluster import OPTICS, DBSCAN
from sklearn.preprocessing import StandardScaler
import hdbscan


def list_dataset_pairs(experiment_dir, max_datasets=None):
    experiment_dir = Path(experiment_dir)
    if not experiment_dir.exists():
        raise FileNotFoundError(f"Папка эксперимента не найдена: {experiment_dir}")

    pairs = []
    for combo_dir in sorted([p for p in experiment_dir.iterdir() if p.is_dir()]):
        for x_path in sorted(combo_dir.glob("rep*_X.npy")):
            rep = x_path.stem.replace("_X", "")
            rf_path = combo_dir / f"{rep}_rf.npy"
            if rf_path.exists():
                pairs.append(
                    {
                        "combo": combo_dir.name,
                        "rep": rep,
                        "x_path": str(x_path),
                        "rf_path": str(rf_path),
                    }
                )

    if max_datasets is not None:
        pairs = pairs[:max_datasets]

    if not pairs:
        raise ValueError("Не найдены пары rep*_X.npy + rep*_rf.npy")
    return pairs


def _evaluate_optics_dataset(ds, min_samples_grid, xi_grid, min_cluster_size_grid):
    X = np.load(ds["x_path"])
    y_true = np.load(ds["rf_path"])
    Xs = StandardScaler().fit_transform(X)

    best = None
    for min_samples, xi, min_cluster_size in itertools.product(
        min_samples_grid, xi_grid, min_cluster_size_grid
    ):
        model = OPTICS(
            min_samples=min_samples,
            xi=xi,
            min_cluster_size=min_cluster_size,
            cluster_method="xi",
            n_jobs=1,
        )
        y_pred = model.fit_predict(Xs)
        metrics = evaluate_predictions(y_true, y_pred)
        candidate = {
            **metrics,
            "combo": ds["combo"],
            "rep": ds["rep"],
            "algorithm": "OPTICS",
            "min_samples": min_samples,
            "xi": xi,
            "min_cluster_size": min_cluster_size,
        }
        if best is None or (candidate["ari"], candidate["nmi"]) > (best["ari"], best["nmi"]):
            best = candidate
    return best


def _evaluate_dbscan_dataset(ds, eps_grid, min_samples_grid):
    X = np.load(ds["x_path"])
    y_true = np.load(ds["rf_path"])
    Xs = StandardScaler().fit_transform(X)

    best = None
    for eps, min_samples in itertools.product(eps_grid, min_samples_grid):
        model = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=1)
        y_pred = model.fit_predict(Xs)
        metrics = evaluate_predictions(y_true, y_pred)
        candidate = {
            **metrics,
            "combo": ds["combo"],
            "rep": ds["rep"],
            "algorithm": "DBSCAN",
            "eps": eps,
            "min_samples": min_samples,
        }
        if best is None or (candidate["ari"], candidate["nmi"]) > (best["ari"], best["nmi"]):
            best = candidate
    return best


def _evaluate_hdbscan_dataset(ds, min_cluster_size_grid, min_samples_grid):
    import hdbscan

    X = np.load(ds["x_path"])
    y_true = np.load(ds["rf_path"])
    Xs = StandardScaler().fit_transform(X)

    best = None
    for min_cluster_size, min_samples in itertools.product(min_cluster_size_grid, min_samples_grid):
        model = hdbscan.HDBSCAN(
            min_cluster_size=int(min_cluster_size),
            min_samples=None if min_samples is None else int(min_samples),
            cluster_selection_method="eom",
            core_dist_n_jobs=1,
        )
        y_pred = model.fit_predict(Xs)
        metrics = evaluate_predictions(y_true, y_pred)
        candidate = {
            **metrics,
            "combo": ds["combo"],
            "rep": ds["rep"],
            "algorithm": "HDBSCAN",
            "min_cluster_size": min_cluster_size,
            "min_samples": min_samples,
        }
        if best is None or (candidate["ari"], candidate["nmi"]) > (best["ari"], best["nmi"]):
            best = candidate
    return best


def _run_algorithm_parallel(name, datasets, worker_fn, worker_kwargs, out_dir, n_jobs):
    print(f"\n{name}: start on {len(datasets)} datasets, n_jobs={n_jobs}")
    rows = Parallel(n_jobs=n_jobs, backend="loky", verbose=0)(
        delayed(worker_fn)(ds, **worker_kwargs) for ds in datasets
    )
    df = pd.DataFrame(rows)

    # Сводка по комбинациям (как раньше) + сводка по ARI/NMI
    summary_ari = summarize_results(df, score_col="ari")
    summary_nmi = summarize_results(df, score_col="nmi")
    summary = summary_ari.merge(summary_nmi, on="combo", how="left")

    detail_path = out_dir / f"{name.lower()}_best_per_dataset.csv"
    summary_path = out_dir / f"{name.lower()}_summary_by_combo.csv"
    df.to_csv(detail_path, index=False)
    summary.to_csv(summary_path, index=False)

    print(f"{name}: done")
    print(f"Saved: {detail_path}")
    print(f"Saved: {summary_path}")
    return df, summary


def run_all_density_benchmarks_fast(
    experiment_dir,
    save_dir="benchmark_results_fast",
    n_jobs=-1,
    mode="quick",
    run_hdbscan=True,
    max_datasets=None,
):
    
    if n_jobs == -1:
        cpu_cnt = os.cpu_count() or 4
        n_jobs = max(1, cpu_cnt - 1)

    out_dir = Path(save_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    datasets = list_dataset_pairs(experiment_dir, max_datasets=max_datasets)
    print(f"Total datasets: {len(datasets)}")

    if mode not in {"quick", "full"}:
        raise ValueError("mode должен быть 'quick' или 'full'")

    if mode == "quick":
        optics_kwargs = {
            "min_samples_grid": (10,),
            "xi_grid": (0.05,),
            "min_cluster_size_grid": (0.05, 0.1),
        }
        dbscan_kwargs = {
            "eps_grid": (0.8, 1.2, 1.6, 2.0),
            "min_samples_grid": (5, 10),
        }
        hdbscan_kwargs = {
            "min_cluster_size_grid": (10, 20),
            "min_samples_grid": (None, 10),
        }
    else:
        optics_kwargs = {
            "min_samples_grid": (5, 10, 15),
            "xi_grid": (0.03, 0.05, 0.1),
            "min_cluster_size_grid": (0.03, 0.05, 0.1),
        }
        dbscan_kwargs = {
            "eps_grid": tuple(np.round(np.arange(0.6, 3.05, 0.2), 2)),
            "min_samples_grid": (4, 5, 6, 8, 10, 12, 15, 20),
        }
        hdbscan_kwargs = {
            "min_cluster_size_grid": (5, 10, 20, 40),
            "min_samples_grid": (None, 5, 10, 20),
        }

    optics_df, optics_summary = _run_algorithm_parallel(
        name="OPTICS",
        datasets=datasets,
        worker_fn=_evaluate_optics_dataset,
        worker_kwargs=optics_kwargs,
        out_dir=out_dir,
        n_jobs=n_jobs,
    )

    dbscan_df, dbscan_summary = _run_algorithm_parallel(
        name="DBSCAN",
        datasets=datasets,
        worker_fn=_evaluate_dbscan_dataset,
        worker_kwargs=dbscan_kwargs,
        out_dir=out_dir,
        n_jobs=n_jobs,
    )

    hdbscan_df = pd.DataFrame()
    hdbscan_summary = pd.DataFrame()
    if run_hdbscan:
        try:
            hdbscan_df, hdbscan_summary = _run_algorithm_parallel(
                name="HDBSCAN",
                datasets=datasets,
                worker_fn=_evaluate_hdbscan_dataset,
                worker_kwargs=hdbscan_kwargs,
                out_dir=out_dir,
                n_jobs=n_jobs,
            )
        except ImportError:
            print("HDBSCAN пропущен: пакет hdbscan не установлен")

    # Общая сравнительная таблица по алгоритмам
    combined = pd.concat([optics_df, dbscan_df, hdbscan_df], ignore_index=True)
    summary_algorithms = (
        combined.groupby("algorithm")
        .agg(
            ari_mean=("ari", "mean"),
            ari_std=("ari", "std"),
            nmi_mean=("nmi", "mean"),
            nmi_std=("nmi", "std"),
            pred_clusters_mean=("pred_clusters", "mean"),
            noise_points_mean=("noise_points", "mean"),
        )
        .reset_index()
        .sort_values(["ari_mean", "nmi_mean"], ascending=False)
    )

    combined_path = out_dir / "all_algorithms_detailed.csv"
    summary_path = out_dir / "all_algorithms_summary.csv"
    combined.to_csv(combined_path, index=False)
    summary_algorithms.to_csv(summary_path, index=False)

    print("\nAll done")
    print(f"Saved: {combined_path}")
    print(f"Saved: {summary_path}")
    display(summary_algorithms)

    return combined, summary_algorithms, optics_summary, dbscan_summary, hdbscan_summary


# Быстрый запуск на всех 540 датасетах:
# combined_df, summary_df, optics_sum, dbscan_sum, hdbscan_sum = run_all_density_benchmarks_fast(
#     experiment_dir="synthetic_datasets/experiment",
#     save_dir="benchmark_results_fast",
#     n_jobs=-1,
#     mode="quick",      # quick быстрее, full тщательнее
#     run_hdbscan=True,
# )

In [ ]:
def run_dbscan_parameter_sweep_540(
    experiment_dir,
    save_dir="benchmark_results_dbscan_sweep",
    eps_grid=None,
    min_samples_grid=(4, 5, 6, 8, 10, 12, 15, 20),
    n_jobs=-1,
    progress_every=30,
):
    if eps_grid is None:
        # Широкая сетка eps для стандартизованных данных
        eps_grid = tuple(np.round(np.arange(0.6, 3.05, 0.2), 2))

    if n_jobs == -1:
        cpu_cnt = os.cpu_count() or 4
        n_jobs = max(1, cpu_cnt - 1)

    out_dir = Path(save_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    datasets = list_dataset_pairs(experiment_dir)
    total_datasets = len(datasets)
    grid_size = len(eps_grid) * len(min_samples_grid)
    print(f"Datasets: {total_datasets}")
    print(f"Grid size: {grid_size} (eps={len(eps_grid)} x min_samples={len(min_samples_grid)})")

    def _process_dataset(ds):
        X = np.load(ds["x_path"])
        y_true = np.load(ds["rf_path"])
        Xs = StandardScaler().fit_transform(X)

        rows = []
        best_row = None

        for eps, min_samples in itertools.product(eps_grid, min_samples_grid):
            model = DBSCAN(eps=float(eps), min_samples=int(min_samples), n_jobs=1)
            y_pred = model.fit_predict(Xs)
            m = evaluate_predictions(y_true, y_pred)

            row = {
                "combo": ds["combo"],
                "rep": ds["rep"],
                "algorithm": "DBSCAN",
                "eps": float(eps),
                "min_samples": int(min_samples),
                "ari": m["ari"],
                "nmi": m["nmi"],
                "pred_clusters": m["pred_clusters"],
                "noise_points": m["noise_points"],
            }
            rows.append(row)

            if best_row is None or (row["ari"], row["nmi"]) > (best_row["ari"], best_row["nmi"]):
                best_row = row

        return rows, best_row

    processed = 0
    all_rows = []
    best_per_dataset = []

    # Запускаем параллельно по датасетам 
    results = Parallel(n_jobs=n_jobs, backend="loky", verbose=0)(
        delayed(_process_dataset)(ds) for ds in datasets
    )

    for rows, best_row in results:
        all_rows.extend(rows)
        best_per_dataset.append(best_row)
        processed += 1
        if processed == 1 or processed % progress_every == 0 or processed == total_datasets:
            print(f"DBSCAN_SWEEP: {processed}/{total_datasets}")

    all_df = pd.DataFrame(all_rows)
    best_dataset_df = pd.DataFrame(best_per_dataset)

    # Лучшая пара параметров для каждой combo (агрегируем по 30 репликам)
    combo_param_agg = (
        all_df.groupby(["combo", "eps", "min_samples"], as_index=False)
        .agg(
            ari_mean=("ari", "mean"),
            ari_std=("ari", "std"),
            nmi_mean=("nmi", "mean"),
            nmi_std=("nmi", "std"),
            pred_clusters_mean=("pred_clusters", "mean"),
            noise_points_mean=("noise_points", "mean"),
        )
    )

    combo_best = (
        combo_param_agg.sort_values(["combo", "ari_mean", "nmi_mean"], ascending=[True, False, False])
        .groupby("combo", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

    # Лучшая пара параметров глобально по всем датасетам
    global_param_agg = (
        all_df.groupby(["eps", "min_samples"], as_index=False)
        .agg(
            ari_mean=("ari", "mean"),
            ari_std=("ari", "std"),
            nmi_mean=("nmi", "mean"),
            nmi_std=("nmi", "std"),
            pred_clusters_mean=("pred_clusters", "mean"),
            noise_points_mean=("noise_points", "mean"),
        )
        .sort_values(["ari_mean", "nmi_mean"], ascending=False)
        .reset_index(drop=True)
    )

    all_path = out_dir / "dbscan_sweep_all_results.csv"
    best_dataset_path = out_dir / "dbscan_sweep_best_per_dataset.csv"
    combo_best_path = out_dir / "dbscan_sweep_best_params_by_combo.csv"
    global_best_path = out_dir / "dbscan_sweep_global_best_params.csv"

    all_df.to_csv(all_path, index=False)
    best_dataset_df.to_csv(best_dataset_path, index=False)
    combo_best.to_csv(combo_best_path, index=False)
    global_param_agg.to_csv(global_best_path, index=False)

    print("\nDBSCAN sweep done")
    print(f"Saved: {all_path}")
    print(f"Saved: {best_dataset_path}")
    print(f"Saved: {combo_best_path}")
    print(f"Saved: {global_best_path}")

    print("\nЛучшие параметры по каждой combo:")
    display(combo_best.sort_values("ari_mean", ascending=False))

    print("\nТоп-10 параметров глобально:")
    display(global_param_agg.head(10))

    return all_df, best_dataset_df, combo_best, global_param_agg


In [10]:
all_df, best_dataset_df, combo_best_df, global_best_df = run_dbscan_parameter_sweep_540(
    experiment_dir="synthetic_datasets/experiment",
    save_dir="benchmark_results_dbscan_sweep",
    eps_grid=tuple(np.round(np.arange(0.6, 3.05, 0.2), 2)),
    min_samples_grid=(4, 5, 6, 8, 10, 12, 15, 20),
    n_jobs=-1,
    progress_every=30,
)

Datasets: 540
Grid size: 104 (eps=13 x min_samples=8)
DBSCAN_SWEEP: 1/540
DBSCAN_SWEEP: 30/540
DBSCAN_SWEEP: 60/540
DBSCAN_SWEEP: 90/540
DBSCAN_SWEEP: 120/540
DBSCAN_SWEEP: 150/540
DBSCAN_SWEEP: 180/540
DBSCAN_SWEEP: 210/540
DBSCAN_SWEEP: 240/540
DBSCAN_SWEEP: 270/540
DBSCAN_SWEEP: 300/540
DBSCAN_SWEEP: 330/540
DBSCAN_SWEEP: 360/540
DBSCAN_SWEEP: 390/540
DBSCAN_SWEEP: 420/540
DBSCAN_SWEEP: 450/540
DBSCAN_SWEEP: 480/540
DBSCAN_SWEEP: 510/540
DBSCAN_SWEEP: 540/540

DBSCAN sweep done
Saved: benchmark_results_dbscan_sweep/dbscan_sweep_all_results.csv
Saved: benchmark_results_dbscan_sweep/dbscan_sweep_best_per_dataset.csv
Saved: benchmark_results_dbscan_sweep/dbscan_sweep_best_params_by_combo.csv
Saved: benchmark_results_dbscan_sweep/dbscan_sweep_global_best_params.csv

Лучшие параметры по каждой combo:


,combo,eps,min_samples,ari_mean,ari_std,nmi_mean,nmi_std,pred_clusters_mean,noise_points_mean
0,V=15_K= 7_alpha=0.25,1.4,4,1.000000,0.000000,1.000000,0.000000,7.000000,0.000000
1,V=15_K= 7_alpha=0.50,2.0,4,1.000000,0.000000,1.000000,0.000000,7.000000,0.000000
16,V=50_K=21_alpha=0.50,3.0,4,1.000000,0.000000,1.000000,0.000000,21.000000,0.000000
15,V=50_K=21_alpha=0.25,2.0,4,1.000000,0.000000,1.000000,0.000000,21.000000,0.000000
13,V=50_K=15_alpha=0.50,3.0,4,1.000000,0.000000,1.000000,0.000000,15.000000,0.000000
12,V=50_K=15_alpha=0.25,2.2,4,1.000000,0.000000,1.000000,0.000000,15.000000,0.000000
9,V=50_K= 7_alpha=0.25,2.4,4,1.000000,0.000000,1.000000,0.000000,7.000000,0.000000
6,V=15_K=21_alpha=0.25,1.2,4,1.000000,0.000000,1.000000,0.000000,21.000000,0.000000
4,V=15_K=15_alpha=0.50,1.8,4,1.000000,0.000000,1.000000,0.000000,15.000000,0.000000
3,V=15_K=15_alpha=0.25,1.4,4,1.000000,0.000000,1.000000,0.000000,15.000000,0.000000



Топ-10 параметров глобально:


,eps,min_samples,ari_mean,ari_std,nmi_mean,nmi_std,pred_clusters_mean,noise_points_mean
0,2.4,5,0.712914,0.378995,0.781064,0.361878,10.898148,188.740741
1,2.4,4,0.712906,0.379346,0.781165,0.361764,10.894444,187.557407
2,2.4,6,0.711877,0.378180,0.780989,0.361674,10.912963,190.188889
3,2.4,8,0.710214,0.377305,0.780378,0.361760,10.951852,193.314815
4,2.4,10,0.709392,0.374810,0.780440,0.361464,11.003704,196.903704
5,2.4,12,0.708680,0.373120,0.780374,0.361802,11.035185,201.327778
6,2.4,15,0.705092,0.373832,0.778864,0.362523,11.087037,208.818519
7,2.6,20,0.697734,0.398941,0.773257,0.368309,10.381481,173.475926
8,2.4,20,0.692782,0.379607,0.772412,0.364309,11.051852,224.177778
9,2.6,15,0.692486,0.406595,0.767548,0.370920,10.275926,171.779630


In [ ]:
def run_hdbscan_parameter_sweep_540(
    experiment_dir,
    save_dir="benchmark_results_hdbscan_sweep",
    min_cluster_size_grid=(5, 8, 10, 12, 15, 20, 30, 40, 60),
    min_samples_grid=(None, 4, 5, 6, 8, 10, 12, 15, 20),
    n_jobs=-1,
    progress_every=30,
):
    try:
        import hdbscan
    except ImportError as exc:
        raise ImportError("Пакет hdbscan не установлен. Выполните: %pip install hdbscan") from exc

    if n_jobs == -1:
        cpu_cnt = os.cpu_count() or 4
        n_jobs = max(1, cpu_cnt - 1)

    out_dir = Path(save_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    datasets = list_dataset_pairs(experiment_dir)
    total_datasets = len(datasets)
    grid_size = len(min_cluster_size_grid) * len(min_samples_grid)
    print(f"Datasets: {total_datasets}")
    print(
        f"Grid size: {grid_size} (min_cluster_size={len(min_cluster_size_grid)} x "
        f"min_samples={len(min_samples_grid)})"
    )

    def _process_dataset(ds):
        X = np.load(ds["x_path"])
        y_true = np.load(ds["rf_path"])
        Xs = StandardScaler().fit_transform(X)

        rows = []
        best_row = None

        for min_cluster_size, min_samples in itertools.product(min_cluster_size_grid, min_samples_grid):
            model = hdbscan.HDBSCAN(
                min_cluster_size=int(min_cluster_size),
                min_samples=None if min_samples is None else int(min_samples),
                cluster_selection_method="eom",
                core_dist_n_jobs=1,
            )
            y_pred = model.fit_predict(Xs)
            m = evaluate_predictions(y_true, y_pred)

            row = {
                "combo": ds["combo"],
                "rep": ds["rep"],
                "algorithm": "HDBSCAN",
                "min_cluster_size": int(min_cluster_size),
                "min_samples": min_samples,
                "ari": m["ari"],
                "nmi": m["nmi"],
                "pred_clusters": m["pred_clusters"],
                "noise_points": m["noise_points"],
            }
            rows.append(row)

            if best_row is None or (row["ari"], row["nmi"]) > (best_row["ari"], best_row["nmi"]):
                best_row = row

        return rows, best_row

    processed = 0
    all_rows = []
    best_per_dataset = []

    results = Parallel(n_jobs=n_jobs, backend="loky", verbose=0)(
        delayed(_process_dataset)(ds) for ds in datasets
    )

    for rows, best_row in results:
        all_rows.extend(rows)
        best_per_dataset.append(best_row)
        processed += 1
        if processed == 1 or processed % progress_every == 0 or processed == total_datasets:
            print(f"HDBSCAN_SWEEP: {processed}/{total_datasets}")

    all_df = pd.DataFrame(all_rows)
    best_dataset_df = pd.DataFrame(best_per_dataset)

    # Лучшая пара параметров для каждой combo
    combo_param_agg = (
        all_df.groupby(["combo", "min_cluster_size", "min_samples"], as_index=False, dropna=False)
        .agg(
            ari_mean=("ari", "mean"),
            ari_std=("ari", "std"),
            nmi_mean=("nmi", "mean"),
            nmi_std=("nmi", "std"),
            pred_clusters_mean=("pred_clusters", "mean"),
            noise_points_mean=("noise_points", "mean"),
        )
    )

    combo_best = (
        combo_param_agg.sort_values(["combo", "ari_mean", "nmi_mean"], ascending=[True, False, False])
        .groupby("combo", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

    # Лучшая пара параметров глобально
    global_param_agg = (
        all_df.groupby(["min_cluster_size", "min_samples"], as_index=False, dropna=False)
        .agg(
            ari_mean=("ari", "mean"),
            ari_std=("ari", "std"),
            nmi_mean=("nmi", "mean"),
            nmi_std=("nmi", "std"),
            pred_clusters_mean=("pred_clusters", "mean"),
            noise_points_mean=("noise_points", "mean"),
        )
        .sort_values(["ari_mean", "nmi_mean"], ascending=False)
        .reset_index(drop=True)
    )

    all_path = out_dir / "hdbscan_sweep_all_results.csv"
    best_dataset_path = out_dir / "hdbscan_sweep_best_per_dataset.csv"
    combo_best_path = out_dir / "hdbscan_sweep_best_params_by_combo.csv"
    global_best_path = out_dir / "hdbscan_sweep_global_best_params.csv"

    all_df.to_csv(all_path, index=False)
    best_dataset_df.to_csv(best_dataset_path, index=False)
    combo_best.to_csv(combo_best_path, index=False)
    global_param_agg.to_csv(global_best_path, index=False)

    print("\nHDBSCAN sweep done")
    print(f"Saved: {all_path}")
    print(f"Saved: {best_dataset_path}")
    print(f"Saved: {combo_best_path}")
    print(f"Saved: {global_best_path}")

    print("\nЛучшие параметры по каждой combo:")
    display(combo_best.sort_values("ari_mean", ascending=False))

    print("\nТоп-10 параметров глобально:")
    display(global_param_agg.head(10))

    return all_df, best_dataset_df, combo_best, global_param_agg


In [12]:

h_all_df, h_best_dataset_df, h_combo_best_df, h_global_best_df = run_hdbscan_parameter_sweep_540(
    experiment_dir="synthetic_datasets/experiment",
    save_dir="benchmark_results_hdbscan_sweep",
    min_cluster_size_grid=(5, 8, 10, 12, 15, 20, 30, 40, 60),
    min_samples_grid=(None, 4, 5, 6, 8, 10, 12, 15, 20),
    n_jobs=-1,
    progress_every=30,
)

Datasets: 540
Grid size: 81 (min_cluster_size=9 x min_samples=9)
HDBSCAN_SWEEP: 1/540
HDBSCAN_SWEEP: 30/540
HDBSCAN_SWEEP: 60/540
HDBSCAN_SWEEP: 90/540
HDBSCAN_SWEEP: 120/540
HDBSCAN_SWEEP: 150/540
HDBSCAN_SWEEP: 180/540
HDBSCAN_SWEEP: 210/540
HDBSCAN_SWEEP: 240/540
HDBSCAN_SWEEP: 270/540
HDBSCAN_SWEEP: 300/540
HDBSCAN_SWEEP: 330/540
HDBSCAN_SWEEP: 360/540
HDBSCAN_SWEEP: 390/540
HDBSCAN_SWEEP: 420/540
HDBSCAN_SWEEP: 450/540
HDBSCAN_SWEEP: 480/540
HDBSCAN_SWEEP: 510/540
HDBSCAN_SWEEP: 540/540

HDBSCAN sweep done
Saved: benchmark_results_hdbscan_sweep/hdbscan_sweep_all_results.csv
Saved: benchmark_results_hdbscan_sweep/hdbscan_sweep_best_per_dataset.csv
Saved: benchmark_results_hdbscan_sweep/hdbscan_sweep_best_params_by_combo.csv
Saved: benchmark_results_hdbscan_sweep/hdbscan_sweep_global_best_params.csv

Лучшие параметры по каждой combo:


,combo,min_cluster_size,min_samples,ari_mean,ari_std,nmi_mean,nmi_std,pred_clusters_mean,noise_points_mean
0,V=15_K= 7_alpha=0.25,5,4.0,1.000000,0.000000,1.000000,6.837661e-17,7.000000,0.0
1,V=15_K= 7_alpha=0.50,5,4.0,1.000000,0.000000,1.000000,8.500328e-17,7.000000,0.0
16,V=50_K=21_alpha=0.50,5,4.0,1.000000,0.000000,1.000000,1.009989e-16,21.000000,0.0
15,V=50_K=21_alpha=0.25,5,4.0,1.000000,0.000000,1.000000,1.009989e-16,21.000000,0.0
14,V=50_K=15_alpha=0.75,5,4.0,1.000000,0.000000,1.000000,1.254042e-16,15.000000,0.0
13,V=50_K=15_alpha=0.50,5,4.0,1.000000,0.000000,1.000000,9.219900e-17,15.000000,0.0
12,V=50_K=15_alpha=0.25,5,4.0,1.000000,0.000000,1.000000,1.071256e-16,15.000000,0.0
11,V=50_K= 7_alpha=0.75,5,4.0,1.000000,0.000000,1.000000,9.887242e-17,7.000000,0.0
10,V=50_K= 7_alpha=0.50,5,4.0,1.000000,0.000000,1.000000,5.049947e-17,7.000000,0.0
9,V=50_K= 7_alpha=0.25,5,4.0,1.000000,0.000000,1.000000,6.837661e-17,7.000000,0.0



Топ-10 параметров глобально:


,min_cluster_size,min_samples,ari_mean,ari_std,nmi_mean,nmi_std,pred_clusters_mean,noise_points_mean
0,5,4.0,0.967478,0.092870,0.984682,0.045737,14.118519,11.703704
1,8,4.0,0.967459,0.092935,0.984675,0.045770,14.111111,11.487037
2,10,4.0,0.967342,0.093184,0.984667,0.045785,14.105556,11.409259
3,12,4.0,0.967013,0.094419,0.984544,0.046137,14.087037,11.072222
4,15,4.0,0.966724,0.095301,0.984511,0.046209,14.077778,10.959259
5,20,4.0,0.966239,0.096728,0.984331,0.046615,14.037037,10.333333
6,12,5.0,0.962409,0.105984,0.982459,0.050927,14.074074,13.400000
7,10,5.0,0.962228,0.106618,0.982465,0.050891,14.088889,13.709259
8,8,5.0,0.962186,0.106732,0.982467,0.050888,14.090741,13.746296
9,15,5.0,0.962134,0.106985,0.982393,0.051108,14.059259,13.187037


In [ ]:
def run_optics_parameter_sweep_540(
    experiment_dir,
    save_dir="benchmark_results_optics_sweep",
    min_samples_grid=(4, 5, 6, 8, 10, 12, 15, 20),
    xi_grid=(0.02, 0.03, 0.05, 0.08, 0.1),
    min_cluster_size_grid=(0.03, 0.05, 0.08, 0.1, 0.15),
    n_jobs=-1,
    progress_every=30,
):
    if n_jobs == -1:
        cpu_cnt = os.cpu_count() or 4
        n_jobs = max(1, cpu_cnt - 1)

    out_dir = Path(save_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    datasets = list_dataset_pairs(experiment_dir)
    total_datasets = len(datasets)
    grid_size = len(min_samples_grid) * len(xi_grid) * len(min_cluster_size_grid)
    print(f"Datasets: {total_datasets}")
    print(
        f"Grid size: {grid_size} (min_samples={len(min_samples_grid)} x "
        f"xi={len(xi_grid)} x min_cluster_size={len(min_cluster_size_grid)})"
    )

    def _process_dataset(ds):
        X = np.load(ds["x_path"])
        y_true = np.load(ds["rf_path"])
        Xs = StandardScaler().fit_transform(X)

        rows = []
        best_row = None

        for min_samples, xi, min_cluster_size in itertools.product(
            min_samples_grid, xi_grid, min_cluster_size_grid
        ):
            model = OPTICS(
                min_samples=int(min_samples),
                xi=float(xi),
                min_cluster_size=float(min_cluster_size),
                cluster_method="xi",
                n_jobs=1,
            )
            y_pred = model.fit_predict(Xs)
            m = evaluate_predictions(y_true, y_pred)

            row = {
                "combo": ds["combo"],
                "rep": ds["rep"],
                "algorithm": "OPTICS",
                "min_samples": int(min_samples),
                "xi": float(xi),
                "min_cluster_size": float(min_cluster_size),
                "ari": m["ari"],
                "nmi": m["nmi"],
                "pred_clusters": m["pred_clusters"],
                "noise_points": m["noise_points"],
            }
            rows.append(row)

            if best_row is None or (row["ari"], row["nmi"]) > (best_row["ari"], best_row["nmi"]):
                best_row = row

        return rows, best_row

    processed = 0
    all_rows = []
    best_per_dataset = []

    results = Parallel(n_jobs=n_jobs, backend="loky", verbose=0)(
        delayed(_process_dataset)(ds) for ds in datasets
    )

    for rows, best_row in results:
        all_rows.extend(rows)
        best_per_dataset.append(best_row)
        processed += 1
        if processed == 1 or processed % progress_every == 0 or processed == total_datasets:
            print(f"OPTICS_SWEEP: {processed}/{total_datasets}")

    all_df = pd.DataFrame(all_rows)
    best_dataset_df = pd.DataFrame(best_per_dataset)

    # Лучшая тройка параметров для каждой combo
    combo_param_agg = (
        all_df.groupby(["combo", "min_samples", "xi", "min_cluster_size"], as_index=False)
        .agg(
            ari_mean=("ari", "mean"),
            ari_std=("ari", "std"),
            nmi_mean=("nmi", "mean"),
            nmi_std=("nmi", "std"),
            pred_clusters_mean=("pred_clusters", "mean"),
            noise_points_mean=("noise_points", "mean"),
        )
    )

    combo_best = (
        combo_param_agg.sort_values(["combo", "ari_mean", "nmi_mean"], ascending=[True, False, False])
        .groupby("combo", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

    # Лучшая тройка параметров глобально
    global_param_agg = (
        all_df.groupby(["min_samples", "xi", "min_cluster_size"], as_index=False)
        .agg(
            ari_mean=("ari", "mean"),
            ari_std=("ari", "std"),
            nmi_mean=("nmi", "mean"),
            nmi_std=("nmi", "std"),
            pred_clusters_mean=("pred_clusters", "mean"),
            noise_points_mean=("noise_points", "mean"),
        )
        .sort_values(["ari_mean", "nmi_mean"], ascending=False)
        .reset_index(drop=True)
    )

    all_path = out_dir / "optics_sweep_all_results.csv"
    best_dataset_path = out_dir / "optics_sweep_best_per_dataset.csv"
    combo_best_path = out_dir / "optics_sweep_best_params_by_combo.csv"
    global_best_path = out_dir / "optics_sweep_global_best_params.csv"

    all_df.to_csv(all_path, index=False)
    best_dataset_df.to_csv(best_dataset_path, index=False)
    combo_best.to_csv(combo_best_path, index=False)
    global_param_agg.to_csv(global_best_path, index=False)

    print("\nOPTICS sweep done")
    print(f"Saved: {all_path}")
    print(f"Saved: {best_dataset_path}")
    print(f"Saved: {combo_best_path}")
    print(f"Saved: {global_best_path}")

    print("\nЛучшие параметры по каждой combo:")
    display(combo_best.sort_values("ari_mean", ascending=False))

    print("\nТоп-10 параметров глобально:")
    display(global_param_agg.head(10))

    return all_df, best_dataset_df, combo_best, global_param_agg

In [14]:
o_all_df, o_best_dataset_df, o_combo_best_df, o_global_best_df = run_optics_parameter_sweep_540(
    experiment_dir="synthetic_datasets/experiment",
    save_dir="benchmark_results_optics_sweep",
    min_samples_grid=(4, 5, 6, 8, 10, 12, 15, 20),
    xi_grid=(0.02, 0.03, 0.05, 0.08, 0.1),
    min_cluster_size_grid=(0.03, 0.05, 0.08, 0.1, 0.15),
    n_jobs=-1,
    progress_every=30,
)

Datasets: 540
Grid size: 200 (min_samples=8 x xi=5 x min_cluster_size=5)
OPTICS_SWEEP: 1/540
OPTICS_SWEEP: 30/540
OPTICS_SWEEP: 60/540
OPTICS_SWEEP: 90/540
OPTICS_SWEEP: 120/540
OPTICS_SWEEP: 150/540
OPTICS_SWEEP: 180/540
OPTICS_SWEEP: 210/540
OPTICS_SWEEP: 240/540
OPTICS_SWEEP: 270/540
OPTICS_SWEEP: 300/540
OPTICS_SWEEP: 330/540
OPTICS_SWEEP: 360/540
OPTICS_SWEEP: 390/540
OPTICS_SWEEP: 420/540
OPTICS_SWEEP: 450/540
OPTICS_SWEEP: 480/540
OPTICS_SWEEP: 510/540
OPTICS_SWEEP: 540/540

OPTICS sweep done
Saved: benchmark_results_optics_sweep/optics_sweep_all_results.csv
Saved: benchmark_results_optics_sweep/optics_sweep_best_per_dataset.csv
Saved: benchmark_results_optics_sweep/optics_sweep_best_params_by_combo.csv
Saved: benchmark_results_optics_sweep/optics_sweep_global_best_params.csv

Лучшие параметры по каждой combo:


,combo,min_samples,xi,min_cluster_size,ari_mean,ari_std,nmi_mean,nmi_std,pred_clusters_mean,noise_points_mean
0,V=15_K= 7_alpha=0.25,4,0.08,0.03,1.000000,0.000000,1.000000,8.246530e-17,7.000000,0.000000
1,V=15_K= 7_alpha=0.50,4,0.08,0.03,1.000000,0.000000,1.000000,7.713922e-17,7.000000,0.000000
16,V=50_K=21_alpha=0.50,4,0.08,0.03,1.000000,0.000000,1.000000,9.447587e-17,21.000000,0.000000
15,V=50_K=21_alpha=0.25,4,0.08,0.03,1.000000,0.000000,1.000000,9.219900e-17,21.000000,0.000000
14,V=50_K=15_alpha=0.75,4,0.05,0.03,1.000000,0.000000,1.000000,9.219900e-17,15.000000,0.000000
13,V=50_K=15_alpha=0.50,4,0.05,0.03,1.000000,0.000000,1.000000,9.669913e-17,15.000000,0.000000
12,V=50_K=15_alpha=0.25,4,0.08,0.03,1.000000,0.000000,1.000000,1.287489e-16,15.000000,0.000000
11,V=50_K= 7_alpha=0.75,4,0.05,0.03,1.000000,0.000000,1.000000,8.500328e-17,7.000000,0.000000
10,V=50_K= 7_alpha=0.50,4,0.05,0.03,1.000000,0.000000,1.000000,8.746766e-17,7.000000,0.000000
9,V=50_K= 7_alpha=0.25,4,0.05,0.03,1.000000,0.000000,1.000000,9.219900e-17,7.000000,0.000000



Топ-10 параметров глобально:


,min_samples,xi,min_cluster_size,ari_mean,ari_std,nmi_mean,nmi_std,pred_clusters_mean,noise_points_mean
0,20,0.02,0.03,0.967594,0.086624,0.987240,0.031866,14.016667,31.757407
1,15,0.02,0.03,0.961815,0.098638,0.985177,0.036744,14.022222,33.977778
2,12,0.02,0.03,0.956041,0.102044,0.982224,0.039082,14.042593,37.077778
3,12,0.03,0.03,0.953530,0.134329,0.984022,0.051324,13.805556,35.675926
4,15,0.03,0.03,0.952327,0.139405,0.983579,0.051631,13.764815,39.912963
5,10,0.03,0.03,0.951957,0.141563,0.982313,0.065049,13.809259,36.672222
6,8,0.03,0.03,0.951803,0.136553,0.982686,0.052090,13.831481,38.350000
7,10,0.02,0.03,0.950044,0.107864,0.979635,0.042367,14.064815,41.970370
8,6,0.03,0.03,0.947978,0.135393,0.979752,0.058180,13.868519,39.890741
9,5,0.03,0.03,0.946404,0.131205,0.978578,0.050547,13.894444,41.846296


In [ ]:
from pathlib import Path
import pandas as pd


def build_final_combo_comparison(
    dbscan_dir="benchmark_results_dbscan_sweep",
    hdbscan_dir="benchmark_results_hdbscan_sweep",
    optics_dir="benchmark_results_optics_sweep",
    save_path="benchmark_results_final/final_best_params_comparison_by_combo.csv",
):
    # Читает *_best_params_by_combo.csv для DBSCAN/HDBSCAN/OPTICS и строит единую сравнительную таблицу по combo.

    dbscan_path = Path(dbscan_dir) / "dbscan_sweep_best_params_by_combo.csv"
    hdbscan_path = Path(hdbscan_dir) / "hdbscan_sweep_best_params_by_combo.csv"
    optics_path = Path(optics_dir) / "optics_sweep_best_params_by_combo.csv"

    for p in [dbscan_path, hdbscan_path, optics_path]:
        if not p.exists():
            raise FileNotFoundError(f"Файл не найден: {p}")

    db = pd.read_csv(dbscan_path)
    hb = pd.read_csv(hdbscan_path)
    op = pd.read_csv(optics_path)

    db["algorithm"] = "DBSCAN"
    hb["algorithm"] = "HDBSCAN"
    op["algorithm"] = "OPTICS"

    # Унифицируем столбцы параметров (чтобы можно было смотреть одной таблицей)
    for df in [db, hb, op]:
        if "eps" not in df.columns:
            df["eps"] = pd.NA
        if "min_samples" not in df.columns:
            df["min_samples"] = pd.NA
        if "min_cluster_size" not in df.columns:
            df["min_cluster_size"] = pd.NA
        if "xi" not in df.columns:
            df["xi"] = pd.NA

    keep_cols = [
        "algorithm",
        "combo",
        "ari_mean",
        "ari_std",
        "nmi_mean",
        "nmi_std",
        "pred_clusters_mean",
        "noise_points_mean",
        "eps",
        "min_samples",
        "min_cluster_size",
        "xi",
    ]

    final_df = pd.concat([
        db[keep_cols],
        hb[keep_cols],
        op[keep_cols],
    ], ignore_index=True)

    final_df = final_df.sort_values(["combo", "ari_mean", "nmi_mean"], ascending=[True, False, False])

    # Победитель по combo
    winners_df = final_df.groupby("combo", as_index=False).head(1).reset_index(drop=True)

    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    final_df.to_csv(save_path, index=False)
    winners_path = save_path.parent / "final_best_algorithm_per_combo.csv"
    winners_df.to_csv(winners_path, index=False)

    print(f"Saved: {save_path}")
    print(f"Saved: {winners_path}")

    print("\nЕдиная сравнительная таблица (все алгоритмы по каждой combo):")
    display(final_df)

    print("\nЛучший алгоритм по каждой combo:")
    display(winners_df)

    return final_df, winners_df

final_df, winners_df = build_final_combo_comparison()

Saved: benchmark_results_final/final_best_params_comparison_by_combo.csv
Saved: benchmark_results_final/final_best_algorithm_per_combo.csv

Единая сравнительная таблица (все алгоритмы по каждой combo):


/var/folders/yv/h7q0tn6n6_x4n12sbqbwbbnw0000gn/T/ipykernel_450/3221272477.py:57: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_df = pd.concat([


,algorithm,combo,ari_mean,ari_std,nmi_mean,nmi_std,pred_clusters_mean,noise_points_mean,eps,min_samples,min_cluster_size,xi
0,DBSCAN,V=15_K= 7_alpha=0.25,1.000000,0.000000,1.000000,0.000000e+00,7.000000,0.000000,1.4,4.0,NaN,NaN
18,HDBSCAN,V=15_K= 7_alpha=0.25,1.000000,0.000000,1.000000,6.837661e-17,7.000000,0.000000,NaN,4.0,5.00,NaN
36,OPTICS,V=15_K= 7_alpha=0.25,1.000000,0.000000,1.000000,8.246530e-17,7.000000,0.000000,NaN,4.0,0.03,0.08
1,DBSCAN,V=15_K= 7_alpha=0.50,1.000000,0.000000,1.000000,0.000000e+00,7.000000,0.000000,2.0,4.0,NaN,NaN
19,HDBSCAN,V=15_K= 7_alpha=0.50,1.000000,0.000000,1.000000,8.500328e-17,7.000000,0.000000,NaN,4.0,5.00,NaN
37,OPTICS,V=15_K= 7_alpha=0.50,1.000000,0.000000,1.000000,7.713922e-17,7.000000,0.000000,NaN,4.0,0.03,0.08
38,OPTICS,V=15_K= 7_alpha=0.75,0.890467,0.162514,0.945428,7.407163e-02,5.866667,148.166667,NaN,20.0,0.08,0.02
20,HDBSCAN,V=15_K= 7_alpha=0.75,0.868564,0.174934,0.914009,1.187208e-01,6.400000,36.500000,NaN,4.0,5.00,NaN
2,DBSCAN,V=15_K= 7_alpha=0.75,0.859022,0.098197,0.915198,5.261351e-02,6.333333,38.000000,2.2,8.0,NaN,NaN
3,DBSCAN,V=15_K=15_alpha=0.25,1.000000,0.000000,1.000000,0.000000e+00,15.000000,0.000000,1.4,4.0,NaN,NaN



Лучший алгоритм по каждой combo:


,algorithm,combo,ari_mean,ari_std,nmi_mean,nmi_std,pred_clusters_mean,noise_points_mean,eps,min_samples,min_cluster_size,xi
0,DBSCAN,V=15_K= 7_alpha=0.25,1.000000,0.000000,1.000000,0.000000e+00,7.000000,0.000000,1.4,4.0,NaN,NaN
1,DBSCAN,V=15_K= 7_alpha=0.50,1.000000,0.000000,1.000000,0.000000e+00,7.000000,0.000000,2.0,4.0,NaN,NaN
2,OPTICS,V=15_K= 7_alpha=0.75,0.890467,0.162514,0.945428,7.407163e-02,5.866667,148.166667,NaN,20.0,0.08,0.02
3,DBSCAN,V=15_K=15_alpha=0.25,1.000000,0.000000,1.000000,0.000000e+00,15.000000,0.000000,1.4,4.0,NaN,NaN
4,DBSCAN,V=15_K=15_alpha=0.50,1.000000,0.000000,1.000000,0.000000e+00,15.000000,0.000000,1.8,4.0,NaN,NaN
5,OPTICS,V=15_K=15_alpha=0.75,0.860060,0.126751,0.950679,3.438373e-02,13.100000,126.466667,NaN,15.0,0.05,0.02
6,DBSCAN,V=15_K=21_alpha=0.25,1.000000,0.000000,1.000000,0.000000e+00,21.000000,0.000000,1.2,4.0,NaN,NaN
7,OPTICS,V=15_K=21_alpha=0.50,1.000000,0.000000,1.000000,1.009989e-16,21.000000,0.000000,NaN,10.0,0.03,0.05
8,OPTICS,V=15_K=21_alpha=0.75,0.771730,0.140719,0.925245,3.096440e-02,18.200000,152.233333,NaN,20.0,0.03,0.02
9,DBSCAN,V=50_K= 7_alpha=0.25,1.000000,0.000000,1.000000,0.000000e+00,7.000000,0.000000,2.4,4.0,NaN,NaN


Кластеризация идет по признакам

In [ ]:
# Переопределенная версия с параметрами, адаптированными для кластеризации ПО ПРИЗНАКАМ

def _make_feature_labels_from_object_labels(X, y_obj):
    """
    Псевдо-истинные метки признаков: для каждого признака берем индекс кластера объектов,
    в котором среднее значение признака максимально.
    """
    unique_clusters = np.sort(np.unique(y_obj))
    class_means = np.vstack([X[y_obj == c].mean(axis=0) for c in unique_clusters])  # shape: K x V
    # Для каждого признака j выбираем кластер c с максимальным средним значением
    feature_labels_idx = np.argmax(class_means, axis=0)
    return unique_clusters[feature_labels_idx]

def _evaluate_dbscan_features_dataset(ds, eps_grid, min_samples_grid):
    X = np.load(ds["x_path"])
    y_obj = np.load(ds["rf_path"])

    X_feat = StandardScaler().fit_transform(X.T)
    y_feat_true = _make_feature_labels_from_object_labels(X, y_obj)
    n_feat = X_feat.shape[0]

    valid_min_samples = [int(ms) for ms in min_samples_grid if int(ms) <= n_feat]
    if not valid_min_samples:
        raise ValueError(f"DBSCAN(features): нет допустимых min_samples для n_features={n_feat}")

    best = None
    for eps, min_samples in itertools.product(eps_grid, valid_min_samples):
        model = DBSCAN(eps=float(eps), min_samples=int(min_samples), n_jobs=1)
        y_feat_pred = model.fit_predict(X_feat)
        m = evaluate_predictions(y_feat_true, y_feat_pred)

        candidate = {
            **m,
            "combo": ds["combo"],
            "rep": ds["rep"],
            "algorithm": "DBSCAN_FEATURES",
            "eps": float(eps),
            "min_samples": int(min_samples),
        }
        if best is None or (candidate["ari"], candidate["nmi"]) > (best["ari"], best["nmi"]):
            best = candidate

    return best


def _evaluate_hdbscan_features_dataset(ds, min_cluster_size_grid, min_samples_grid):
    X = np.load(ds["x_path"])
    y_obj = np.load(ds["rf_path"])

    X_feat = StandardScaler().fit_transform(X.T)
    y_feat_true = _make_feature_labels_from_object_labels(X, y_obj)
    n_feat = X_feat.shape[0]

    valid_pairs = []
    for mcs, ms in itertools.product(min_cluster_size_grid, min_samples_grid):
        mcs_i = int(mcs)
        ms_i = None if ms is None else int(ms)
        if mcs_i <= n_feat and (ms_i is None or ms_i <= n_feat):
            valid_pairs.append((mcs_i, ms_i))

    if not valid_pairs:
        raise ValueError(f"HDBSCAN(features): нет допустимых параметров для n_features={n_feat}")

    best = None
    for min_cluster_size, min_samples in valid_pairs:
        model = hdbscan.HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            cluster_selection_method="eom",
            core_dist_n_jobs=1,
        )
        y_feat_pred = model.fit_predict(X_feat)
        m = evaluate_predictions(y_feat_true, y_feat_pred)

        candidate = {
            **m,
            "combo": ds["combo"],
            "rep": ds["rep"],
            "algorithm": "HDBSCAN_FEATURES",
            "min_cluster_size": min_cluster_size,
            "min_samples": min_samples,
        }
        if best is None or (candidate["ari"], candidate["nmi"]) > (best["ari"], best["nmi"]):
            best = candidate

    return best


def _evaluate_optics_features_dataset(ds, min_samples_grid, xi_grid, min_cluster_size_grid):
    X = np.load(ds["x_path"])
    y_obj = np.load(ds["rf_path"])

    X_feat = StandardScaler().fit_transform(X.T)
    y_feat_true = _make_feature_labels_from_object_labels(X, y_obj)
    n_feat = X_feat.shape[0]

    valid_min_samples = [int(ms) for ms in min_samples_grid if int(ms) <= n_feat]
    if not valid_min_samples:
        raise ValueError(f"OPTICS(features): нет допустимых min_samples для n_features={n_feat}")

    best = None
    for min_samples, xi, min_cluster_size in itertools.product(
        valid_min_samples, xi_grid, min_cluster_size_grid
    ):
        # Для OPTICS: если min_cluster_size >= 2, передаем как int;
        # float допустим только в диапазоне (0, 1].
        mcs_value = int(min_cluster_size) if float(min_cluster_size) >= 2 else float(min_cluster_size)

        model = OPTICS(
            min_samples=int(min_samples),
            xi=float(xi),
            min_cluster_size=mcs_value,
            cluster_method="xi",
            n_jobs=1,
        )
        y_feat_pred = model.fit_predict(X_feat)
        m = evaluate_predictions(y_feat_true, y_feat_pred)

        candidate = {
            **m,
            "combo": ds["combo"],
            "rep": ds["rep"],
            "algorithm": "OPTICS_FEATURES",
            "min_samples": int(min_samples),
            "xi": float(xi),
            "min_cluster_size": float(min_cluster_size),
        }
        if best is None or (candidate["ari"], candidate["nmi"]) > (best["ari"], best["nmi"]):
            best = candidate

    return best

def run_feature_clustering_density_sweeps(
    experiment_dir,
    save_dir="benchmark_results_features",
    n_jobs=-1,
    progress_every=30,
):
    # Полный прогон DBSCAN/HDBSCAN/OPTICS по ПРИЗНАКАМ (X.T) с параметрами, адаптированными под малое число samples (V=15 или V=50).

    if n_jobs == -1:
        cpu_cnt = os.cpu_count() or 4
        n_jobs = max(1, cpu_cnt - 1)

    out_dir = Path(save_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    datasets = list_dataset_pairs(experiment_dir)
    total = len(datasets)

    # Адаптированные сетки под feature-space
    dbscan_kwargs = {
        "eps_grid": (2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 15.0, 18.0, 22.0, 26.0, 30.0, 35.0, 40.0),
        "min_samples_grid": (2, 3, 4, 5, 6, 8, 10),
    }
    hdbscan_kwargs = {
        "min_cluster_size_grid": (2, 3, 4, 5, 6, 8, 10, 12),
        "min_samples_grid": (None, 1, 2, 3, 4, 5, 6, 8),
    }
    optics_kwargs = {
        "min_samples_grid": (2, 3, 4, 5, 6, 8, 10, 12),
        "xi_grid": (0.005, 0.01, 0.02, 0.03, 0.05, 0.08),
        "min_cluster_size_grid": (2, 3, 4, 5, 6, 8, 10, 12),
    }

    def _run_one(name, fn, kwargs):
        rows = Parallel(n_jobs=n_jobs, backend="loky", verbose=0)(
            delayed(fn)(ds, **kwargs) for ds in datasets
        )
        df = pd.DataFrame(rows)

        summary_ari = summarize_results(df, score_col="ari")
        summary_nmi = summarize_results(df, score_col="nmi")
        summary = summary_ari.merge(summary_nmi, on="combo", how="left")

        df.to_csv(out_dir / f"{name}_features_best_per_dataset.csv", index=False)
        summary.to_csv(out_dir / f"{name}_features_summary_by_combo.csv", index=False)
        print(f"{name}: done")
        return df, summary

    print(f"Feature clustering start: {total} datasets")
    db_df, db_sum = _run_one("dbscan", _evaluate_dbscan_features_dataset, dbscan_kwargs)
    hb_df, hb_sum = _run_one("hdbscan", _evaluate_hdbscan_features_dataset, hdbscan_kwargs)
    op_df, op_sum = _run_one("optics", _evaluate_optics_features_dataset, optics_kwargs)

    final = pd.concat([db_df, hb_df, op_df], ignore_index=True)
    final_summary = (
        final.groupby("algorithm")
        .agg(
            ari_mean=("ari", "mean"),
            ari_std=("ari", "std"),
            nmi_mean=("nmi", "mean"),
            nmi_std=("nmi", "std"),
            pred_clusters_mean=("pred_clusters", "mean"),
            noise_points_mean=("noise_points", "mean"),
        )
        .reset_index()
        .sort_values(["ari_mean", "nmi_mean"], ascending=False)
    )

    final.to_csv(out_dir / "all_algorithms_features_detailed.csv", index=False)
    final_summary.to_csv(out_dir / "all_algorithms_features_summary.csv", index=False)

    print("Feature clustering sweeps done")
    display(final_summary)
    return final, final_summary, db_sum, hb_sum, op_sum


In [ ]:
features_df, features_summary_df, db_feat_sum, hb_feat_sum, op_feat_sum = run_feature_clustering_density_sweeps(
    experiment_dir="synthetic_datasets/experiment",
    save_dir="benchmark_results_features",
    n_jobs=-1,
)

Feature clustering start: 540 datasets


dbscan: done
hdbscan: done
optics: done
Feature clustering sweeps done


,algorithm,ari_mean,ari_std,nmi_mean,nmi_std,pred_clusters_mean,noise_points_mean
2,OPTICS_FEATURES,0.103089,0.092996,0.421851,0.158381,3.388889,15.462963
1,HDBSCAN_FEATURES,0.079651,0.085460,0.402018,0.181761,3.935185,13.235185
0,DBSCAN_FEATURES,0.074527,0.078441,0.335161,0.149987,2.316667,15.368519


In [ ]:
def analyze_feature_clustering_results_no_nan(save_dir="benchmark_results_features"):
    out_dir = Path(save_dir)

    db_path = out_dir / "dbscan_features_best_per_dataset.csv"
    hb_path = out_dir / "hdbscan_features_best_per_dataset.csv"
    op_path = out_dir / "optics_features_best_per_dataset.csv"

    for p in [db_path, hb_path, op_path]:
        if not p.exists():
            raise FileNotFoundError(f"Файл не найден: {p}")

    db_det = pd.read_csv(db_path)
    hb_det = pd.read_csv(hb_path)
    op_det = pd.read_csv(op_path)

    all_det = pd.concat([db_det, hb_det, op_det], ignore_index=True)

    # Агрегация по (algorithm, combo) из детальных данных
    combo_summary = (
        all_det.groupby(["algorithm", "combo"], as_index=False)
        .agg(
            ari_mean=("ari", "mean"),
            ari_std=("ari", "std"),
            nmi_mean=("nmi", "mean"),
            nmi_std=("nmi", "std"),
            pred_clusters_mean=("pred_clusters", "mean"),
            noise_points_mean=("noise_points", "mean"),
        )
    )

    # Добавим V/K/alpha для дополнительных срезов
    parsed = combo_summary["combo"].apply(_parse_combo_fields)
    combo_summary = pd.concat([combo_summary, parsed], axis=1)

    winners_by_combo = (
        combo_summary.sort_values(["combo", "ari_mean", "nmi_mean"], ascending=[True, False, False])
        .groupby("combo", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

    summary_by_alpha = (
        combo_summary.groupby(["algorithm", "alpha"], as_index=False)
        .agg(
            ari_mean=("ari_mean", "mean"),
            nmi_mean=("nmi_mean", "mean"),
            pred_clusters_mean=("pred_clusters_mean", "mean"),
            noise_points_mean=("noise_points_mean", "mean"),
        )
        .sort_values(["alpha", "ari_mean"], ascending=[True, False])
    )

    summary_by_V = (
        combo_summary.groupby(["algorithm", "V"], as_index=False)
        .agg(
            ari_mean=("ari_mean", "mean"),
            nmi_mean=("nmi_mean", "mean"),
            pred_clusters_mean=("pred_clusters_mean", "mean"),
            noise_points_mean=("noise_points_mean", "mean"),
        )
        .sort_values(["V", "ari_mean"], ascending=[True, False])
    )

    combo_path = out_dir / "features_combo_all_algorithms_no_nan.csv"
    winners_path = out_dir / "features_winner_per_combo_no_nan.csv"
    alpha_path = out_dir / "features_summary_by_alpha_no_nan.csv"
    v_path = out_dir / "features_summary_by_V_no_nan.csv"

    combo_summary.to_csv(combo_path, index=False)
    winners_by_combo.to_csv(winners_path, index=False)
    summary_by_alpha.to_csv(alpha_path, index=False)
    summary_by_V.to_csv(v_path, index=False)

    print(f"Saved: {combo_path}")
    print(f"Saved: {winners_path}")
    print(f"Saved: {alpha_path}")
    print(f"Saved: {v_path}")

    print("\nСводка по всем алгоритмам и combo:")
    display(combo_summary[["algorithm", "combo", "ari_mean", "nmi_mean", "pred_clusters_mean", "noise_points_mean"]])

    print("\nЛучший алгоритм по каждой combo:")
    display(winners_by_combo[["combo", "algorithm", "ari_mean", "nmi_mean", "pred_clusters_mean", "noise_points_mean"]])

    print("\nКраткая таблица по alpha:")
    display(summary_by_alpha)

    print("\nКраткая таблица по V:")
    display(summary_by_V)

    return combo_summary, winners_by_combo, summary_by_alpha, summary_by_V

combo_summary_no_nan, winners_no_nan, by_alpha_no_nan, by_V_no_nan = analyze_feature_clustering_results_no_nan(
    save_dir="benchmark_results_features"
)

Saved: benchmark_results_features/features_combo_all_algorithms_no_nan.csv
Saved: benchmark_results_features/features_winner_per_combo_no_nan.csv
Saved: benchmark_results_features/features_summary_by_alpha_no_nan.csv
Saved: benchmark_results_features/features_summary_by_V_no_nan.csv

Сводка по всем алгоритмам и combo:


,algorithm,combo,ari_mean,nmi_mean,pred_clusters_mean,noise_points_mean
0,DBSCAN_FEATURES,V=15_K= 7_alpha=0.25,0.184463,0.465374,1.866667,5.333333
1,DBSCAN_FEATURES,V=15_K= 7_alpha=0.50,0.207932,0.458603,1.866667,6.700000
2,DBSCAN_FEATURES,V=15_K= 7_alpha=0.75,0.140258,0.406558,1.733333,5.633333
3,DBSCAN_FEATURES,V=15_K=15_alpha=0.25,0.092630,0.426500,1.566667,7.033333
4,DBSCAN_FEATURES,V=15_K=15_alpha=0.50,0.053603,0.350261,1.300000,7.700000
5,DBSCAN_FEATURES,V=15_K=15_alpha=0.75,0.072293,0.415166,1.433333,7.466667
6,DBSCAN_FEATURES,V=15_K=21_alpha=0.25,0.070159,0.431074,1.366667,6.700000
7,DBSCAN_FEATURES,V=15_K=21_alpha=0.50,0.049187,0.369265,1.200000,7.066667
8,DBSCAN_FEATURES,V=15_K=21_alpha=0.75,0.042301,0.362809,1.266667,8.133333
9,DBSCAN_FEATURES,V=50_K= 7_alpha=0.25,0.107638,0.317611,4.566667,17.666667



Лучший алгоритм по каждой combo:


,combo,algorithm,ari_mean,nmi_mean,pred_clusters_mean,noise_points_mean
0,V=15_K= 7_alpha=0.25,OPTICS_FEATURES,0.213065,0.534348,2.666667,4.966667
1,V=15_K= 7_alpha=0.50,OPTICS_FEATURES,0.243578,0.530319,2.566667,4.866667
2,V=15_K= 7_alpha=0.75,OPTICS_FEATURES,0.194821,0.476671,2.133333,5.600000
3,V=15_K=15_alpha=0.25,OPTICS_FEATURES,0.125161,0.494542,1.966667,7.000000
4,V=15_K=15_alpha=0.50,OPTICS_FEATURES,0.088654,0.472975,1.900000,6.700000
5,V=15_K=15_alpha=0.75,OPTICS_FEATURES,0.102406,0.490832,1.933333,8.066667
6,V=15_K=21_alpha=0.25,OPTICS_FEATURES,0.098873,0.509401,2.166667,5.366667
7,V=15_K=21_alpha=0.50,OPTICS_FEATURES,0.077379,0.476333,2.200000,5.700000
8,V=15_K=21_alpha=0.75,OPTICS_FEATURES,0.080673,0.495746,2.166667,6.000000
9,V=50_K= 7_alpha=0.25,OPTICS_FEATURES,0.139578,0.372235,5.300000,19.133333



Краткая таблица по alpha:


,algorithm,alpha,ari_mean,nmi_mean,pred_clusters_mean,noise_points_mean
6,OPTICS_FEATURES,0.25,0.109603,0.429692,3.422222,15.283333
3,HDBSCAN_FEATURES,0.25,0.087223,0.422270,4.088889,12.483333
0,DBSCAN_FEATURES,0.25,0.084503,0.355329,2.388889,14.261111
7,OPTICS_FEATURES,0.50,0.102997,0.419808,3.411111,14.950000
4,HDBSCAN_FEATURES,0.50,0.080451,0.400540,4.022222,13.155556
1,DBSCAN_FEATURES,0.50,0.075828,0.322172,2.088889,15.305556
8,OPTICS_FEATURES,0.75,0.096666,0.416053,3.333333,16.155556
5,HDBSCAN_FEATURES,0.75,0.071280,0.383243,3.694444,14.066667
2,DBSCAN_FEATURES,0.75,0.063250,0.327981,2.472222,16.538889



Краткая таблица по V:


,algorithm,V,ari_mean,nmi_mean,pred_clusters_mean,noise_points_mean
4,OPTICS_FEATURES,15.0,0.136068,0.497907,2.188889,6.029630
0,DBSCAN_FEATURES,15.0,0.101425,0.409512,1.511111,6.862963
2,HDBSCAN_FEATURES,15.0,0.099550,0.445707,2.251852,6.100000
5,OPTICS_FEATURES,50.0,0.070110,0.345795,4.588889,24.896296
3,HDBSCAN_FEATURES,50.0,0.059753,0.358328,5.618519,20.370370
1,DBSCAN_FEATURES,50.0,0.047629,0.260809,3.122222,23.874074


In [ ]:
# CONSENSUS-ready feature clustering
# 1) без выбора best внутри датасета (сохраняем все комбинации параметров)
# 2) сохраняем y_feat_pred и как список (labels), и как .npy файл (labels_path)

import json
from pathlib import Path
from joblib import Parallel, delayed
import itertools
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN, OPTICS
from sklearn.preprocessing import StandardScaler
import hdbscan


def _safe_name(s):
    return str(s).replace(" ", "_").replace("/", "-")


def _save_feature_labels(base_labels_dir, algorithm, combo, rep, param_id, y_pred):
    alg_dir = Path(base_labels_dir) / algorithm / _safe_name(combo)
    alg_dir.mkdir(parents=True, exist_ok=True)
    out_path = alg_dir / f"{rep}_{param_id}.npy"
    np.save(out_path, y_pred)
    return str(out_path)


def _feature_dbscan_all_results_for_dataset(ds, eps_grid, min_samples_grid, labels_dir):
    X = np.load(ds["x_path"])
    y_obj = np.load(ds["rf_path"])

    X_feat = StandardScaler().fit_transform(X.T)
    y_feat_true = _make_feature_labels_from_object_labels(X, y_obj)
    n_feat = X_feat.shape[0]

    valid_min_samples = [int(ms) for ms in min_samples_grid if int(ms) <= n_feat]
    rows = []

    for eps, min_samples in itertools.product(eps_grid, valid_min_samples):
        model = DBSCAN(eps=float(eps), min_samples=int(min_samples), n_jobs=1)
        y_feat_pred = model.fit_predict(X_feat)
        m = evaluate_predictions(y_feat_true, y_feat_pred)

        param_id = f"eps{float(eps):.4f}_ms{int(min_samples)}"
        labels_path = _save_feature_labels(labels_dir, "DBSCAN_FEATURES", ds["combo"], ds["rep"], param_id, y_feat_pred)

        rows.append(
            {
                "combo": ds["combo"],
                "rep": ds["rep"],
                "algorithm": "DBSCAN_FEATURES",
                "eps": float(eps),
                "min_samples": int(min_samples),
                "min_cluster_size": np.nan,
                "xi": np.nan,
                "ari": m["ari"],
                "nmi": m["nmi"],
                "pred_clusters": m["pred_clusters"],
                "noise_points": m["noise_points"],
                "labels": json.dumps(y_feat_pred.tolist()),
                "labels_path": labels_path,
            }
        )

    return rows


def _feature_hdbscan_all_results_for_dataset(ds, min_cluster_size_grid, min_samples_grid, labels_dir):
    X = np.load(ds["x_path"])
    y_obj = np.load(ds["rf_path"])

    X_feat = StandardScaler().fit_transform(X.T)
    y_feat_true = _make_feature_labels_from_object_labels(X, y_obj)
    n_feat = X_feat.shape[0]

    valid_pairs = []
    for mcs, ms in itertools.product(min_cluster_size_grid, min_samples_grid):
        mcs_i = int(mcs)
        ms_i = None if ms is None else int(ms)
        if mcs_i <= n_feat and (ms_i is None or ms_i <= n_feat):
            valid_pairs.append((mcs_i, ms_i))

    rows = []
    for min_cluster_size, min_samples in valid_pairs:
        model = hdbscan.HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            cluster_selection_method="eom",
            core_dist_n_jobs=1,
        )
        y_feat_pred = model.fit_predict(X_feat)
        m = evaluate_predictions(y_feat_true, y_feat_pred)

        ms_tag = "None" if min_samples is None else str(min_samples)
        param_id = f"mcs{min_cluster_size}_ms{ms_tag}"
        labels_path = _save_feature_labels(labels_dir, "HDBSCAN_FEATURES", ds["combo"], ds["rep"], param_id, y_feat_pred)

        rows.append(
            {
                "combo": ds["combo"],
                "rep": ds["rep"],
                "algorithm": "HDBSCAN_FEATURES",
                "eps": np.nan,
                "min_samples": min_samples,
                "min_cluster_size": min_cluster_size,
                "xi": np.nan,
                "ari": m["ari"],
                "nmi": m["nmi"],
                "pred_clusters": m["pred_clusters"],
                "noise_points": m["noise_points"],
                "labels": json.dumps(y_feat_pred.tolist()),
                "labels_path": labels_path,
            }
        )

    return rows


def _feature_optics_all_results_for_dataset(ds, min_samples_grid, xi_grid, min_cluster_size_grid, labels_dir):
    X = np.load(ds["x_path"])
    y_obj = np.load(ds["rf_path"])

    X_feat = StandardScaler().fit_transform(X.T)
    y_feat_true = _make_feature_labels_from_object_labels(X, y_obj)
    n_feat = X_feat.shape[0]

    valid_min_samples = [int(ms) for ms in min_samples_grid if int(ms) <= n_feat]

    rows = []
    for min_samples, xi, min_cluster_size in itertools.product(valid_min_samples, xi_grid, min_cluster_size_grid):
        mcs_value = int(min_cluster_size) if float(min_cluster_size) >= 2 else float(min_cluster_size)
        model = OPTICS(
            min_samples=int(min_samples),
            xi=float(xi),
            min_cluster_size=mcs_value,
            cluster_method="xi",
            n_jobs=1,
        )
        y_feat_pred = model.fit_predict(X_feat)
        m = evaluate_predictions(y_feat_true, y_feat_pred)

        param_id = f"ms{int(min_samples)}_xi{float(xi):.4f}_mcs{str(mcs_value)}"
        labels_path = _save_feature_labels(labels_dir, "OPTICS_FEATURES", ds["combo"], ds["rep"], param_id, y_feat_pred)

        rows.append(
            {
                "combo": ds["combo"],
                "rep": ds["rep"],
                "algorithm": "OPTICS_FEATURES",
                "eps": np.nan,
                "min_samples": int(min_samples),
                "min_cluster_size": mcs_value,
                "xi": float(xi),
                "ari": m["ari"],
                "nmi": m["nmi"],
                "pred_clusters": m["pred_clusters"],
                "noise_points": m["noise_points"],
                "labels": json.dumps(y_feat_pred.tolist()),
                "labels_path": labels_path,
            }
        )

    return rows


def _summarize_feature_all_results(df):
    # Сводка по параметрам внутри combo
    by_combo_params = (
        df.groupby(["combo", "algorithm", "eps", "min_samples", "min_cluster_size", "xi"], dropna=False, as_index=False)
        .agg(
            ari_mean=("ari", "mean"),
            ari_std=("ari", "std"),
            nmi_mean=("nmi", "mean"),
            nmi_std=("nmi", "std"),
            pred_clusters_mean=("pred_clusters", "mean"),
            noise_points_mean=("noise_points", "mean"),
        )
    )

    # Лучшие параметры по combo
    best_by_combo = (
        by_combo_params.sort_values(["combo", "algorithm", "ari_mean", "nmi_mean"], ascending=[True, True, False, False])
        .groupby(["combo", "algorithm"], as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

    # Глобальная сводка по алгоритмам
    algo_summary = (
        df.groupby("algorithm", as_index=False)
        .agg(
            ari_mean=("ari", "mean"),
            ari_std=("ari", "std"),
            nmi_mean=("nmi", "mean"),
            nmi_std=("nmi", "std"),
            pred_clusters_mean=("pred_clusters", "mean"),
            noise_points_mean=("noise_points", "mean"),
        )
        .sort_values(["ari_mean", "nmi_mean"], ascending=False)
    )

    return by_combo_params, best_by_combo, algo_summary


def run_feature_clustering_density_all_results(
    experiment_dir,
    save_dir="benchmark_results_features_consensus_ready",
    n_jobs=-1,
):
    
    if n_jobs == -1:
        cpu_cnt = os.cpu_count() or 4
        n_jobs = max(1, cpu_cnt - 1)

    out_dir = Path(save_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    labels_dir = out_dir / "labels"
    labels_dir.mkdir(parents=True, exist_ok=True)

    datasets = list_dataset_pairs(experiment_dir)

    dbscan_kwargs = {
        "eps_grid": (2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 15.0, 18.0, 22.0, 26.0, 30.0, 35.0, 40.0),
        "min_samples_grid": (2, 3, 4, 5, 6, 8, 10),
    }
    hdbscan_kwargs = {
        "min_cluster_size_grid": (2, 3, 4, 5, 6, 8, 10, 12),
        "min_samples_grid": (None, 1, 2, 3, 4, 5, 6, 8),
    }
    optics_kwargs = {
        "min_samples_grid": (2, 3, 4, 5, 6, 8, 10, 12),
        "xi_grid": (0.005, 0.01, 0.02, 0.03, 0.05, 0.08),
        "min_cluster_size_grid": (2, 3, 4, 5, 6, 8, 10, 12),
    }

    print(f"Feature consensus-ready start: {len(datasets)} datasets")

    db_rows_nested = Parallel(n_jobs=n_jobs, backend="loky", verbose=0)(
        delayed(_feature_dbscan_all_results_for_dataset)(ds, **dbscan_kwargs, labels_dir=labels_dir) for ds in datasets
    )
    hb_rows_nested = Parallel(n_jobs=n_jobs, backend="loky", verbose=0)(
        delayed(_feature_hdbscan_all_results_for_dataset)(ds, **hdbscan_kwargs, labels_dir=labels_dir) for ds in datasets
    )
    op_rows_nested = Parallel(n_jobs=n_jobs, backend="loky", verbose=0)(
        delayed(_feature_optics_all_results_for_dataset)(ds, **optics_kwargs, labels_dir=labels_dir) for ds in datasets
    )

    all_rows = [r for sub in db_rows_nested for r in sub] + [r for sub in hb_rows_nested for r in sub] + [r for sub in op_rows_nested for r in sub]
    all_df = pd.DataFrame(all_rows)

    by_combo_params, best_by_combo, algo_summary = _summarize_feature_all_results(all_df)

    all_path = out_dir / "features_all_results.csv"
    combo_params_path = out_dir / "features_by_combo_params.csv"
    best_combo_path = out_dir / "features_best_params_by_combo_algorithm.csv"
    algo_summary_path = out_dir / "features_algorithm_summary.csv"

    all_df.to_csv(all_path, index=False)
    by_combo_params.to_csv(combo_params_path, index=False)
    best_by_combo.to_csv(best_combo_path, index=False)
    algo_summary.to_csv(algo_summary_path, index=False)

    print(f"Saved: {all_path}")
    print(f"Saved: {combo_params_path}")
    print(f"Saved: {best_combo_path}")
    print(f"Saved: {algo_summary_path}")
    display(algo_summary)

    return all_df, by_combo_params, best_by_combo, algo_summary

In [ ]:
all_feat_df, combo_params_feat_df, best_combo_feat_df, algo_summary_feat_df = run_feature_clustering_density_all_results(
    experiment_dir="synthetic_datasets/experiment",
    save_dir="benchmark_results_features_consensus_ready",
    n_jobs=-1,
)

Feature consensus-ready start: 540 datasets
Saved: benchmark_results_features_consensus_ready/features_all_results.csv
Saved: benchmark_results_features_consensus_ready/features_by_combo_params.csv
Saved: benchmark_results_features_consensus_ready/features_best_params_by_combo_algorithm.csv
Saved: benchmark_results_features_consensus_ready/features_algorithm_summary.csv


,algorithm,ari_mean,ari_std,nmi_mean,nmi_std,pred_clusters_mean,noise_points_mean
2,OPTICS_FEATURES,0.009551,0.036552,0.083963,0.134817,1.303791,9.311343
1,HDBSCAN_FEATURES,0.004953,0.027604,0.047720,0.117244,0.493547,28.998177
0,DBSCAN_FEATURES,0.002588,0.019155,0.024439,0.080875,0.232540,28.188685


In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display


def show_top10_features_all_algorithms(save_dir="benchmark_results_features_consensus_ready"):
    all_path = Path(save_dir) / "features_all_results.csv"
    if not all_path.exists():
        print(f"Файл не найден: {all_path}")
        print("Сначала запустите run_feature_clustering_density_all_results(...)")
        return None

    df = pd.read_csv(all_path)

    algorithms = ["DBSCAN_FEATURES", "HDBSCAN_FEATURES", "OPTICS_FEATURES"]
    cols = [
        "combo", "rep", "algorithm", "ari", "nmi", "pred_clusters", "noise_points",
        "eps", "min_samples", "min_cluster_size", "xi", "labels_path"
    ]

    result = {}
    for alg in algorithms:
        part = df[df["algorithm"] == alg].copy()
        if part.empty:
            print(f"{alg}: нет строк в features_all_results.csv")
            continue

        top10 = (
            part.sort_values(["ari", "nmi"], ascending=[False, False])
            .head(10)
            .reset_index(drop=True)
        )

        show_cols = [c for c in cols if c in top10.columns]
        print(f"TOP-10 {alg} по ARI/NMI")
        display(top10[show_cols])

        result[alg] = top10

    return result

top10_features_all = show_top10_features_all_algorithms("benchmark_results_features_consensus_ready")

TOP-10 DBSCAN_FEATURES по ARI/NMI


,combo,rep,algorithm,ari,nmi,pred_clusters,noise_points,eps,min_samples,min_cluster_size,xi,labels_path
0,V=15_K= 7_alpha=0.50,rep029,DBSCAN_FEATURES,0.536424,0.800000,4,5,30.0,2.0,NaN,NaN,benchmark_results_features_consensus_ready/lab...
1,V=15_K= 7_alpha=0.25,rep004,DBSCAN_FEATURES,0.409639,0.641789,2,5,30.0,5.0,NaN,NaN,benchmark_results_features_consensus_ready/lab...
2,V=15_K= 7_alpha=0.25,rep030,DBSCAN_FEATURES,0.387522,0.614802,2,6,35.0,5.0,NaN,NaN,benchmark_results_features_consensus_ready/lab...
3,V=15_K= 7_alpha=0.50,rep015,DBSCAN_FEATURES,0.378698,0.680552,3,5,30.0,2.0,NaN,NaN,benchmark_results_features_consensus_ready/lab...
4,V=15_K= 7_alpha=0.25,rep003,DBSCAN_FEATURES,0.370899,0.641990,3,3,35.0,2.0,NaN,NaN,benchmark_results_features_consensus_ready/lab...
5,V=15_K= 7_alpha=0.50,rep005,DBSCAN_FEATURES,0.370274,0.614802,3,6,30.0,2.0,NaN,NaN,benchmark_results_features_consensus_ready/lab...
6,V=15_K= 7_alpha=0.50,rep030,DBSCAN_FEATURES,0.365833,0.629466,3,4,35.0,2.0,NaN,NaN,benchmark_results_features_consensus_ready/lab...
7,V=15_K= 7_alpha=0.50,rep009,DBSCAN_FEATURES,0.363972,0.575939,3,4,35.0,2.0,NaN,NaN,benchmark_results_features_consensus_ready/lab...
8,V=15_K= 7_alpha=0.75,rep023,DBSCAN_FEATURES,0.361191,0.686157,4,1,35.0,2.0,NaN,NaN,benchmark_results_features_consensus_ready/lab...
9,V=15_K= 7_alpha=0.75,rep023,DBSCAN_FEATURES,0.359586,0.615291,2,5,35.0,3.0,NaN,NaN,benchmark_results_features_consensus_ready/lab...


TOP-10 HDBSCAN_FEATURES по ARI/NMI


,combo,rep,algorithm,ari,nmi,pred_clusters,noise_points,eps,min_samples,min_cluster_size,xi,labels_path
0,V=15_K= 7_alpha=0.25,rep003,HDBSCAN_FEATURES,0.568611,0.737066,3,2,NaN,1.0,2.0,NaN,benchmark_results_features_consensus_ready/lab...
1,V=15_K= 7_alpha=0.25,rep003,HDBSCAN_FEATURES,0.512106,0.705255,2,3,NaN,NaN,2.0,NaN,benchmark_results_features_consensus_ready/lab...
2,V=15_K= 7_alpha=0.25,rep003,HDBSCAN_FEATURES,0.512106,0.705255,2,3,NaN,2.0,2.0,NaN,benchmark_results_features_consensus_ready/lab...
3,V=15_K= 7_alpha=0.25,rep003,HDBSCAN_FEATURES,0.512106,0.705255,2,3,NaN,2.0,3.0,NaN,benchmark_results_features_consensus_ready/lab...
4,V=15_K= 7_alpha=0.25,rep003,HDBSCAN_FEATURES,0.512106,0.705255,2,3,NaN,2.0,4.0,NaN,benchmark_results_features_consensus_ready/lab...
5,V=15_K= 7_alpha=0.25,rep003,HDBSCAN_FEATURES,0.512106,0.705255,2,3,NaN,2.0,5.0,NaN,benchmark_results_features_consensus_ready/lab...
6,V=15_K= 7_alpha=0.25,rep003,HDBSCAN_FEATURES,0.512106,0.705255,2,3,NaN,2.0,6.0,NaN,benchmark_results_features_consensus_ready/lab...
7,V=15_K= 7_alpha=0.50,rep017,HDBSCAN_FEATURES,0.456621,0.713879,3,2,NaN,1.0,3.0,NaN,benchmark_results_features_consensus_ready/lab...
8,V=15_K= 7_alpha=0.50,rep029,HDBSCAN_FEATURES,0.447732,0.767390,4,1,NaN,1.0,2.0,NaN,benchmark_results_features_consensus_ready/lab...
9,V=15_K=15_alpha=0.25,rep024,HDBSCAN_FEATURES,0.433249,0.741934,4,4,NaN,1.0,2.0,NaN,benchmark_results_features_consensus_ready/lab...


TOP-10 OPTICS_FEATURES по ARI/NMI


,combo,rep,algorithm,ari,nmi,pred_clusters,noise_points,eps,min_samples,min_cluster_size,xi,labels_path
0,V=15_K= 7_alpha=0.50,rep029,OPTICS_FEATURES,0.589202,0.814113,4,4,NaN,2.0,2.0,0.030,benchmark_results_features_consensus_ready/lab...
1,V=15_K= 7_alpha=0.50,rep029,OPTICS_FEATURES,0.589202,0.814113,4,4,NaN,2.0,2.0,0.050,benchmark_results_features_consensus_ready/lab...
2,V=15_K= 7_alpha=0.25,rep003,OPTICS_FEATURES,0.568611,0.737066,3,2,NaN,2.0,2.0,0.005,benchmark_results_features_consensus_ready/lab...
3,V=15_K= 7_alpha=0.50,rep005,OPTICS_FEATURES,0.561404,0.731033,3,4,NaN,2.0,2.0,0.005,benchmark_results_features_consensus_ready/lab...
4,V=15_K= 7_alpha=0.50,rep005,OPTICS_FEATURES,0.561404,0.731033,3,4,NaN,2.0,3.0,0.005,benchmark_results_features_consensus_ready/lab...
5,V=15_K= 7_alpha=0.50,rep005,OPTICS_FEATURES,0.561404,0.731033,3,4,NaN,2.0,2.0,0.010,benchmark_results_features_consensus_ready/lab...
6,V=15_K= 7_alpha=0.50,rep005,OPTICS_FEATURES,0.561404,0.731033,3,4,NaN,2.0,3.0,0.010,benchmark_results_features_consensus_ready/lab...
7,V=15_K= 7_alpha=0.50,rep005,OPTICS_FEATURES,0.561404,0.731033,3,4,NaN,2.0,2.0,0.020,benchmark_results_features_consensus_ready/lab...
8,V=15_K= 7_alpha=0.50,rep005,OPTICS_FEATURES,0.561404,0.731033,3,4,NaN,2.0,3.0,0.020,benchmark_results_features_consensus_ready/lab...
9,V=15_K= 7_alpha=0.50,rep005,OPTICS_FEATURES,0.561404,0.731033,3,4,NaN,2.0,2.0,0.030,benchmark_results_features_consensus_ready/lab...


In [ ]:
# ВИЗУАЛЬНЫЙ ВЫВОД ТАБЛИЦ CONSENSUS-READY РЕЗУЛЬТАТОВ

from pathlib import Path
import pandas as pd
from IPython.display import display


def show_feature_consensus_tables(save_dir="benchmark_results_features_consensus_ready"):
    base = Path(save_dir)

    all_path = base / "features_all_results.csv"
    by_combo_path = base / "features_by_combo_params.csv"
    best_combo_path = base / "features_best_params_by_combo_algorithm.csv"
    algo_summary_path = base / "features_algorithm_summary.csv"

    required = [all_path, by_combo_path, best_combo_path, algo_summary_path]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        print("Не найдены файлы:")
        for p in missing:
            print(" -", p)
        print("Сначала запустите run_feature_clustering_density_all_results(...)")
        return None

    all_df = pd.read_csv(all_path)
    by_combo_df = pd.read_csv(by_combo_path)
    best_combo_df = pd.read_csv(best_combo_path)
    algo_summary_df = pd.read_csv(algo_summary_path)

    pd.set_option("display.max_columns", 200)
    pd.set_option("display.width", 200)

    print("=" * 100)
    print("1) Общая сводка по алгоритмам (главная таблица)")
    print("=" * 100)
    display(algo_summary_df.sort_values(["ari_mean", "nmi_mean"], ascending=False).reset_index(drop=True))

    print("\n" + "=" * 100)
    print("2) Лучшие параметры по каждому combo и алгоритму")
    print("=" * 100)
    cols_best = [
        "combo", "algorithm", "ari_mean", "nmi_mean",
        "pred_clusters_mean", "noise_points_mean",
        "eps", "min_samples", "min_cluster_size", "xi"
    ]
    cols_best = [c for c in cols_best if c in best_combo_df.columns]
    display(best_combo_df[cols_best].sort_values(["combo", "ari_mean", "nmi_mean"], ascending=[True, False, False]).reset_index(drop=True))

    print("\n" + "=" * 100)
    print("3) Победитель (лучший алгоритм) для каждого combo")
    print("=" * 100)
    winner_by_combo = (
        best_combo_df.sort_values(["combo", "ari_mean", "nmi_mean"], ascending=[True, False, False])
        .groupby("combo", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )
    display(winner_by_combo[["combo", "algorithm", "ari_mean", "nmi_mean", "pred_clusters_mean", "noise_points_mean"]])

    print("\n" + "=" * 100)
    print("4) Сколько раз каждый алгоритм победил")
    print("=" * 100)
    win_counts = (
        winner_by_combo["algorithm"]
        .value_counts()
        .rename_axis("algorithm")
        .reset_index(name="wins")
    )
    display(win_counts)

    # Разбор combo -> V, K, alpha
    parts = winner_by_combo["combo"].str.extract(r"V=(\d+)_K=\s*(\d+)_alpha=([0-9.]+)")
    winner_by_combo = winner_by_combo.copy()
    winner_by_combo["V"] = pd.to_numeric(parts[0], errors="coerce")
    winner_by_combo["K"] = pd.to_numeric(parts[1], errors="coerce")
    winner_by_combo["alpha"] = pd.to_numeric(parts[2], errors="coerce")

    print("\n" + "=" * 100)
    print("5) Сводка качества победителей по alpha")
    print("=" * 100)
    by_alpha = (
        winner_by_combo.groupby(["alpha", "algorithm"], as_index=False)
        .agg(
            combos=("combo", "count"),
            ari_mean=("ari_mean", "mean"),
            nmi_mean=("nmi_mean", "mean"),
            pred_clusters_mean=("pred_clusters_mean", "mean"),
            noise_points_mean=("noise_points_mean", "mean"),
        )
        .sort_values(["alpha", "ari_mean", "nmi_mean"], ascending=[True, False, False])
    )
    display(by_alpha)

    print("\n" + "=" * 100)
    print("6) Сводка качества победителей по V")
    print("=" * 100)
    by_v = (
        winner_by_combo.groupby(["V", "algorithm"], as_index=False)
        .agg(
            combos=("combo", "count"),
            ari_mean=("ari_mean", "mean"),
            nmi_mean=("nmi_mean", "mean"),
            pred_clusters_mean=("pred_clusters_mean", "mean"),
            noise_points_mean=("noise_points_mean", "mean"),
        )
        .sort_values(["V", "ari_mean", "nmi_mean"], ascending=[True, False, False])
    )
    display(by_v)


viz_tables = show_feature_consensus_tables("benchmark_results_features_consensus_ready")

1) Общая сводка по алгоритмам (главная таблица)


,algorithm,ari_mean,ari_std,nmi_mean,nmi_std,pred_clusters_mean,noise_points_mean
0,OPTICS_FEATURES,0.009551,0.036552,0.083963,0.134817,1.303791,9.311343
1,HDBSCAN_FEATURES,0.004953,0.027604,0.047720,0.117244,0.493547,28.998177
2,DBSCAN_FEATURES,0.002588,0.019155,0.024439,0.080875,0.232540,28.188685



2) Лучшие параметры по каждому combo и алгоритму


,combo,algorithm,ari_mean,nmi_mean,pred_clusters_mean,noise_points_mean,eps,min_samples,min_cluster_size,xi
0,V=15_K= 7_alpha=0.25,HDBSCAN_FEATURES,0.160919,0.540121,3.433333,3.433333,NaN,1.0,2.0,NaN
1,V=15_K= 7_alpha=0.25,OPTICS_FEATURES,0.155233,0.560402,3.766667,3.966667,NaN,2.0,2.0,0.005
2,V=15_K= 7_alpha=0.25,DBSCAN_FEATURES,0.097554,0.451690,2.600000,5.933333,30.0,2.0,NaN,NaN
3,V=15_K= 7_alpha=0.50,HDBSCAN_FEATURES,0.164240,0.516777,3.433333,2.600000,NaN,1.0,2.0,NaN
4,V=15_K= 7_alpha=0.50,OPTICS_FEATURES,0.154211,0.553085,3.866667,3.433333,NaN,2.0,2.0,0.005
5,V=15_K= 7_alpha=0.50,DBSCAN_FEATURES,0.136555,0.458730,2.466667,7.233333,30.0,2.0,NaN,NaN
6,V=15_K= 7_alpha=0.75,HDBSCAN_FEATURES,0.103535,0.468392,3.000000,2.833333,NaN,1.0,2.0,NaN
7,V=15_K= 7_alpha=0.75,OPTICS_FEATURES,0.098446,0.397708,1.800000,6.833333,NaN,2.0,3.0,0.020
8,V=15_K= 7_alpha=0.75,DBSCAN_FEATURES,0.086680,0.428605,2.400000,4.633333,35.0,2.0,NaN,NaN
9,V=15_K=15_alpha=0.25,HDBSCAN_FEATURES,0.068979,0.543565,3.266667,3.100000,NaN,1.0,2.0,NaN



3) Победитель (лучший алгоритм) для каждого combo


...

,algorithm,wins
0,HDBSCAN_FEATURES,10
1,OPTICS_FEATURES,8



5) Сводка качества победителей по alpha


,alpha,algorithm,combos,ari_mean,nmi_mean,pred_clusters_mean,noise_points_mean
0,0.25,HDBSCAN_FEATURES,4,0.086952,0.464426,5.383333,8.616667
1,0.25,OPTICS_FEATURES,2,0.031453,0.541923,6.800000,13.000000
2,0.50,HDBSCAN_FEATURES,3,0.086266,0.403861,4.988889,11.677778
3,0.50,OPTICS_FEATURES,3,0.027188,0.545446,5.933333,9.411111
4,0.75,HDBSCAN_FEATURES,3,0.070624,0.367302,4.211111,10.933333
5,0.75,OPTICS_FEATURES,3,0.026109,0.528311,5.422222,11.311111



6) Сводка качества победителей по V


,V,algorithm,combos,ari_mean,nmi_mean,pred_clusters_mean,noise_points_mean
0,15,HDBSCAN_FEATURES,4,0.124418,0.517214,3.283333,2.991667
1,15,OPTICS_FEATURES,5,0.034177,0.564463,3.280000,5.180000
2,50,HDBSCAN_FEATURES,6,0.053467,0.350389,6.000000,15.055556
3,50,OPTICS_FEATURES,3,0.017305,0.494268,10.422222,20.755556


При кластеризации по признакам все три плотностных алгоритма работают заметно хуже, чем при кластеризации объектов. Именно поэтому и нужен consensus — чтобы стабилизировать слабые базовые кластеризации и получить надёжное ранжирование признаков.

In [ ]:
# параметры как в кластеризация по объектам 

# def _make_feature_labels_from_object_labels(X, y_obj):
#     """
#     Псевдо-истинные метки признаков: для каждого признака берем индекс кластера объектов,
#     в котором среднее значение признака максимально.
#     """
#     unique_clusters = np.sort(np.unique(y_obj))
#     class_means = np.vstack([X[y_obj == c].mean(axis=0) for c in unique_clusters])  # shape: K x V
#     # Для каждого признака j выбираем кластер c с максимальным средним значением
#     feature_labels_idx = np.argmax(class_means, axis=0)
#     return unique_clusters[feature_labels_idx]

# def _evaluate_dbscan_features_dataset(ds, eps_grid, min_samples_grid):
#     X = np.load(ds["x_path"])
#     y_obj = np.load(ds["rf_path"])

#     X_feat = StandardScaler().fit_transform(X.T)
#     y_feat_true = _make_feature_labels_from_object_labels(X, y_obj)
#     n_feat = X_feat.shape[0]

#     valid_min_samples = [int(ms) for ms in min_samples_grid if int(ms) <= n_feat]
#     if not valid_min_samples:
#         raise ValueError(f"DBSCAN(features): нет допустимых min_samples для n_features={n_feat}")

#     best = None
#     for eps, min_samples in itertools.product(eps_grid, valid_min_samples):
#         model = DBSCAN(eps=float(eps), min_samples=int(min_samples), n_jobs=1)
#         y_feat_pred = model.fit_predict(X_feat)
#         m = evaluate_predictions(y_feat_true, y_feat_pred)

#         candidate = {
#             **m,
#             "combo": ds["combo"],
#             "rep": ds["rep"],
#             "algorithm": "DBSCAN_FEATURES",
#             "eps": float(eps),
#             "min_samples": int(min_samples),
#         }
#         if best is None or (candidate["ari"], candidate["nmi"]) > (best["ari"], best["nmi"]):
#             best = candidate

#     return best


# def _evaluate_hdbscan_features_dataset(ds, min_cluster_size_grid, min_samples_grid):
#     X = np.load(ds["x_path"])
#     y_obj = np.load(ds["rf_path"])

#     X_feat = StandardScaler().fit_transform(X.T)
#     y_feat_true = _make_feature_labels_from_object_labels(X, y_obj)
#     n_feat = X_feat.shape[0]

#     valid_pairs = []
#     for mcs, ms in itertools.product(min_cluster_size_grid, min_samples_grid):
#         mcs_i = int(mcs)
#         ms_i = None if ms is None else int(ms)
#         if mcs_i <= n_feat and (ms_i is None or ms_i <= n_feat):
#             valid_pairs.append((mcs_i, ms_i))

#     if not valid_pairs:
#         raise ValueError(f"HDBSCAN(features): нет допустимых параметров для n_features={n_feat}")

#     best = None
#     for min_cluster_size, min_samples in valid_pairs:
#         model = hdbscan.HDBSCAN(
#             min_cluster_size=min_cluster_size,
#             min_samples=min_samples,
#             cluster_selection_method="eom",
#             core_dist_n_jobs=1,
#         )
#         y_feat_pred = model.fit_predict(X_feat)
#         m = evaluate_predictions(y_feat_true, y_feat_pred)

#         candidate = {
#             **m,
#             "combo": ds["combo"],
#             "rep": ds["rep"],
#             "algorithm": "HDBSCAN_FEATURES",
#             "min_cluster_size": min_cluster_size,
#             "min_samples": min_samples,
#         }
#         if best is None or (candidate["ari"], candidate["nmi"]) > (best["ari"], best["nmi"]):
#             best = candidate

#     return best


# def _evaluate_optics_features_dataset(ds, min_samples_grid, xi_grid, min_cluster_size_grid):
#     X = np.load(ds["x_path"])
#     y_obj = np.load(ds["rf_path"])

#     X_feat = StandardScaler().fit_transform(X.T)
#     y_feat_true = _make_feature_labels_from_object_labels(X, y_obj)
#     n_feat = X_feat.shape[0]

#     valid_min_samples = [int(ms) for ms in min_samples_grid if int(ms) <= n_feat]
#     if not valid_min_samples:
#         raise ValueError(f"OPTICS(features): нет допустимых min_samples для n_features={n_feat}")

#     best = None
#     for min_samples, xi, min_cluster_size in itertools.product(
#         valid_min_samples, xi_grid, min_cluster_size_grid
#     ):
#         model = OPTICS(
#             min_samples=int(min_samples),
#             xi=float(xi),
#             min_cluster_size=float(min_cluster_size),
#             cluster_method="xi",
#             n_jobs=1,
#         )
#         y_feat_pred = model.fit_predict(X_feat)
#         m = evaluate_predictions(y_feat_true, y_feat_pred)

#         candidate = {
#             **m,
#             "combo": ds["combo"],
#             "rep": ds["rep"],
#             "algorithm": "OPTICS_FEATURES",
#             "min_samples": int(min_samples),
#             "xi": float(xi),
#             "min_cluster_size": float(min_cluster_size),
#         }
#         if best is None or (candidate["ari"], candidate["nmi"]) > (best["ari"], best["nmi"]):
#             best = candidate

#     return best

# def run_feature_clustering_density_sweeps(
#     experiment_dir,
#     save_dir="benchmark_results_features",
#     n_jobs=-1,
#     progress_every=30,
# ):
#     """
#     Полный прогон DBSCAN/HDBSCAN/OPTICS по ПРИЗНАКАМ (X.T)
#     с теми же сетками параметров, что и в кластеризации по объектам.
#     """
#     if n_jobs == -1:
#         cpu_cnt = os.cpu_count() or 4
#         n_jobs = max(1, cpu_cnt - 1)

#     out_dir = Path(save_dir)
#     out_dir.mkdir(parents=True, exist_ok=True)

#     datasets = list_dataset_pairs(experiment_dir)
#     total = len(datasets)

#     dbscan_kwargs = {
#         "eps_grid": tuple(np.round(np.arange(0.6, 3.05, 0.2), 2)),
#         "min_samples_grid": (4, 5, 6, 8, 10, 12, 15, 20),
#     }
#     hdbscan_kwargs = {
#         "min_cluster_size_grid": (5, 8, 10, 12, 15, 20, 30, 40, 60),
#         "min_samples_grid": (None, 4, 5, 6, 8, 10, 12, 15, 20),
#     }
#     optics_kwargs = {
#         "min_samples_grid": (4, 5, 6, 8, 10, 12, 15, 20),
#         "xi_grid": (0.02, 0.03, 0.05, 0.08, 0.1),
#         "min_cluster_size_grid": (0.03, 0.05, 0.08, 0.1, 0.15),
#     }

#     def _run_one(name, fn, kwargs):
#         rows = Parallel(n_jobs=n_jobs, backend="loky", verbose=0)(
#             delayed(fn)(ds, **kwargs) for ds in datasets
#         )
#         df = pd.DataFrame(rows)

#         summary_ari = summarize_results(df, score_col="ari")
#         summary_nmi = summarize_results(df, score_col="nmi")
#         summary = summary_ari.merge(summary_nmi, on="combo", how="left")

#         df.to_csv(out_dir / f"{name}_features_best_per_dataset.csv", index=False)
#         summary.to_csv(out_dir / f"{name}_features_summary_by_combo.csv", index=False)
#         print(f"{name}: done")
#         return df, summary

#     print(f"Feature clustering start: {total} datasets")
#     db_df, db_sum = _run_one("dbscan", _evaluate_dbscan_features_dataset, dbscan_kwargs)
#     hb_df, hb_sum = _run_one("hdbscan", _evaluate_hdbscan_features_dataset, hdbscan_kwargs)
#     op_df, op_sum = _run_one("optics", _evaluate_optics_features_dataset, optics_kwargs)

#     final = pd.concat([db_df, hb_df, op_df], ignore_index=True)
#     final_summary = (
#         final.groupby("algorithm")
#         .agg(
#             ari_mean=("ari", "mean"),
#             ari_std=("ari", "std"),
#             nmi_mean=("nmi", "mean"),
#             nmi_std=("nmi", "std"),
#             pred_clusters_mean=("pred_clusters", "mean"),
#             noise_points_mean=("noise_points", "mean"),
#         )
#         .reset_index()
#         .sort_values(["ari_mean", "nmi_mean"], ascending=False)
#     )

#     final.to_csv(out_dir / "all_algorithms_features_detailed.csv", index=False)
#     final_summary.to_csv(out_dir / "all_algorithms_features_summary.csv", index=False)

#     print("Feature clustering sweeps done")
#     display(final_summary)
#     return final, final_summary, db_sum, hb_sum, op_sum


In [ ]:
# features_df, features_summary_df, db_feat_sum, hb_feat_sum, op_feat_sum = run_feature_clustering_density_sweeps(
#     experiment_dir="synthetic_datasets/experiment",
#     save_dir="benchmark_results_features",
#     n_jobs=-1,
# )

Feature clustering start: 540 datasets
dbscan: done
hdbscan: done
optics: done
Feature clustering sweeps done


,algorithm,ari_mean,ari_std,nmi_mean,nmi_std,pred_clusters_mean,noise_points_mean
2,OPTICS_FEATURES,0.046212,0.064684,0.221806,0.163761,1.479630,18.725926
1,HDBSCAN_FEATURES,0.000903,0.006248,0.007071,0.037157,0.075926,31.844444
0,DBSCAN_FEATURES,0.000000,0.000000,0.000000,0.000000,0.000000,32.500000
